# Aereo Water-Body Segmentation — Production SegFormer V3

## End-to-end, reproducible, leakage-controlled, deployment-oriented workflow

This notebook is the **maintained main workflow** for the assignment. It rebuilds the complete
SegFormer solution from raw image–mask pairs and preserves the historical SAM experiments as
comparative research evidence.

The pipeline covers:

- validated image–mask ingestion and portable data registration;
- deterministic split recovery and leakage auditing;
- normalization, synchronized augmentation, and aspect-ratio-safe preprocessing;
- overlapping pixel-space and metadata-preserving GeoTIFF tiling;
- Optuna hyperparameter optimization with a same-code baseline;
- MLflow experiment tracking with a persistent artifact store;
- optional offline Weights & Biases mirroring;
- top-candidate confirmation and random-seed stability;
- resumable final training;
- validation-only probability-threshold calibration;
- frozen held-out test evaluation at original resolution;
- paired statistical comparison, calibration, and performance slices;
- production inference, structured logs, checkpoint integrity verification;
- API smoke tests, deployment round-trip parity, repository tests, and evidence export.

> **Historical boundary:** The original four-experiment notebook remains in the repository for
> zero-shot SAM, fine-tuned SAM, original SegFormer, and SegFormer–SAM hybrid evidence. This
> notebook does not depend on the old private experiment bundle.

## 0. Executive contract

### Primary production decision

SegFormer-B0 remains the production architecture because the previous study showed it to be the
strongest fully automatic model. The V3 study asks a narrower and more defensible question:

> Can a clean, independently reproducible SegFormer pipeline improve or match the historical
> result under controlled HPO, same-code baseline confirmation, validation-only model selection,
> and a frozen held-out test?

### Pre-declared acceptance criteria

The acceptance thresholds are loaded from `configs/acceptance_criteria.yaml` **before** the test
split is opened. They are not modified after final test results become available.

### Test-set firewall

The held-out test split is allowed to exist in the registry, but no test dataframe is materialized
for HPO, confirmation, early stopping, checkpoint selection, seed stability, or threshold
calibration. A model-selection lock file is written before test evaluation begins.

## 1. Assignment compliance map

| Assignment requirement | Notebook section | Repository implementation | Evidence |
|---|---:|---|---|
| Scalable ingestion | 5–6 | `aereo_water.data.manifest` | validated runtime manifest |
| Normalization | 9 | `aereo_water.data.dataset` | tensor statistics and figure |
| Augmentation | 9 | synchronized transforms | paired before/after figure |
| Tiling | 10 | pixel + GeoTIFF tiling | tile manifests and exact reconstruction |
| SOTA PyTorch training | 13–17 | SegFormer trainer | checkpoints and histories |
| HPO | 13 | Optuna TPE + pruning | study DB, trial table, best trial |
| Experiment tracking | 11–18 | MLflow + optional W&B | DB, artifacts, offline runs |
| Data registry | 7 | portable registry | JSON, CSV, checksums |
| Model registry | 25 | portable + MLflow registry | selected model record |
| Inference | 26–27 | `SegFormerPredictor` | mask, overlay, parity |
| Logging | 26–28 | JSONL structured logs | request-level timing |
| API | 28 | FastAPI V3 | health, ready, metadata, upload tests |
| Containerization | repository | `Dockerfile.v3`, Compose V3 | separate Docker validation |
| Comparative analysis | 20–23 | historical A–D + V3 | fair validation-selected comparison |
| Transparent failures | all stages | failure ledger | failure and resolution table |
| CI/CD | 29 | GitHub Actions + tests | compile and pytest evidence |
| Report/presentation | post-run | exported metrics and figures | report-ready evidence bundle |

## 2. Scientific guardrails

1. **HPO objective equals the final quality objective:** original-resolution mean per-image
   validation IoU at a fixed threshold of `0.50`.
2. **No threshold tuning during HPO:** threshold calibration happens once, after final training,
   on validation only.
3. **Same-code control:** the historical SegFormer hyperparameters are always rerun under the V3
   training engine alongside the top Optuna candidates.
4. **Historical SAM prompts:** selected using historical validation rows, then reported on test.
   Test-based prompt selection is prohibited.
5. **Final seed:** fixed before test evaluation. Seed-stability results describe sensitivity but
   do not permit choosing a seed using test performance.
6. **Empty-mask convention:** reported explicitly together with water-present IoU and empty-mask
   false-positive rate.
7. **Latency labels:** model-forward and end-to-end latency are reported separately.
8. **Geospatial honesty:** metadata is preserved when present. Pixel-coordinate demonstrations are
   not presented as georeferenced evidence.
9. **Evidence-derived compliance:** a requirement is marked complete only when its artifact exists
   and passes validation.

## 3. Run controls, stage orchestration, and compute profile

In [ ]:
from pathlib import Path

# ----------------------------- User controls -----------------------------
# Public source notebooks should normally use "smoke". Change this to "full"
# for the final Kaggle experiment. The separate full-run notebook already
# sets RUN_PROFILE="full".
RUN_PROFILE = "smoke"  # smoke | full

# Completed stages are loaded automatically. Set this only when intentionally
# discarding every prior V3 artifact and starting the selected profile clean.
RESET_OUTPUT_ROOT = False

# The repository is the source of truth for reusable logic.
REPOSITORY_URL = (
    "https://github.com/Mshrooom/"
    "Aereo-WaterSeg-DSintern-Assignment.git"
)
REPOSITORY_DIR = Path("/kaggle/working/aereo-water-segmentation")

# Smoke and full evidence use different roots so a completed smoke run can
# never be mistaken for final evidence.
OUTPUT_ROOT = Path(
    f"/kaggle/working/aereo-water-v3-{RUN_PROFILE}"
)
RESUME_ROOT_INPUT = None
# Example after attaching a previous Kaggle output:
# RESUME_ROOT_INPUT = Path("/kaggle/input/aereo-v3-resume/aereo-water-v3-full")

# Tracking policy: MLflow is authoritative; W&B is an optional offline mirror.
WANDB_MODE = "offline"  # offline | online | disabled

# Operational switches.
RUN_NEAR_DUPLICATE_AUDIT = True
RUN_SEED_STABILITY = True
RUN_API_SMOKE_TEST = True
RUN_FULL_PREDICTION_EXPORT = True

In [ ]:
# Execution profiles deliberately separate pipeline validation from final evidence.
PROFILES = {
    "smoke": {
        "completed_hpo_trials": 2,
        "maximum_hpo_attempts": 3,
        "hpo_epochs": 1,
        "hpo_train_images": 64,
        "hpo_validation_images": 32,
        "confirmation_top_k": 1,
        "confirmation_epochs": 1,
        "stability_seeds": [42],
        "stability_epochs": 1,
        "final_epochs": 1,
        "evaluation_limit": 64,
        "minimum_free_disk_gb": 5.0,
    },
    "full": {
        "completed_hpo_trials": 12,
        "maximum_hpo_attempts": 20,
        "hpo_epochs": 4,
        "hpo_train_images": 1000,
        "hpo_validation_images": None,  # all 429 validation images
        "confirmation_top_k": 3,
        "confirmation_epochs": 6,
        "stability_seeds": [42, 2026, 3407],
        "stability_epochs": 4,
        "final_epochs": 15,
        "evaluation_limit": None,       # all 2,841 images
        "minimum_free_disk_gb": 20.0,
    },
}

if RUN_PROFILE not in PROFILES:
    raise ValueError(f"Unknown RUN_PROFILE: {RUN_PROFILE}")

PROFILE = PROFILES[RUN_PROFILE]
PROFILE

### Automatic stage resumption

The workflow always traverses the dependency graph in order:

```text
data → hpo → confirmation → stability → final_train → calibrate
     → evaluate → inference → api_test → export
```

- Completed stages are loaded from their evidence artifacts.
- Missing, failed, or interrupted stages run automatically.
- `stage_state.json` records status, timestamps, and evidence.
- Final training writes `last_state.pt` every epoch and resumes optimizer,
  scheduler, scaler, early-stopping, history, and random-number-generator state.
- `RESET_OUTPUT_ROOT=True` is the only clean-rerun switch. It removes the
  profile-specific output root before execution, avoiding partial stale state.
- HPO fingerprints and integrity checks prevent accidental reuse of incompatible
  study databases or checkpoints.


## 4. Kaggle repository setup and exact environment capture

In [ ]:
import os
import sys
import subprocess
import shutil
from pathlib import Path

def run_command(command, *, cwd=None, check=True):
    """Run a command with visible output and deterministic failure handling."""
    print("$", " ".join(map(str, command)))
    result = subprocess.run(
        list(map(str, command)),
        cwd=cwd,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: {command}"
        )
    return result

# Restore a previous output root or intentionally start clean. These controls
# are mutually exclusive to prevent accidental evidence deletion.
if RESET_OUTPUT_ROOT and RESUME_ROOT_INPUT is not None:
    raise ValueError(
        "RESET_OUTPUT_ROOT and RESUME_ROOT_INPUT cannot be used together."
    )
if RESET_OUTPUT_ROOT and OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

if RESUME_ROOT_INPUT is not None:
    if not Path(RESUME_ROOT_INPUT).exists():
        raise FileNotFoundError(RESUME_ROOT_INPUT)
    if OUTPUT_ROOT.exists():
        shutil.rmtree(OUTPUT_ROOT)
    shutil.copytree(RESUME_ROOT_INPUT, OUTPUT_ROOT)
else:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Clone only when the repository is not already attached or present.
if not REPOSITORY_DIR.exists():
    run_command(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, REPOSITORY_DIR]
    )
else:
    print("Using existing repository:", REPOSITORY_DIR)

# Install the declared production environment and editable repository package.
run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        REPOSITORY_DIR / "requirements" / "production.in",
    ]
)
run_command(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."],
    cwd=REPOSITORY_DIR,
)

In [ ]:
# Imports are deliberately centralized so later cells remain focused on analysis.
import gc
import io
import json
import math
import platform
import random
import shutil
import statistics
import time
import warnings
from dataclasses import asdict
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image
from IPython.display import Markdown, display

import aereo_water
from aereo_water.config import HPOConfig, load_config
from aereo_water.data.dataset import SegmentationTrainingDataset
from aereo_water.data.manifest import (
    assert_split_integrity,
    assign_exact_split,
    discover_pairs,
    make_portable_registry,
    near_duplicate_audit,
    recover_historical_split,
    validate_manifest,
    write_data_registry,
)
from aereo_water.data.tiling import (
    build_multitile_demo,
    reconstruct_geotiff,
    reconstruct_tiles,
    tile_array,
    tile_geotiff,
)
from aereo_water.data.transforms import resize_pair
from aereo_water.evaluation.evaluator import (
    benchmark_model_forward,
    evaluate_manifest,
    summarize_results,
)
from aereo_water.evaluation.metrics import (
    probability_calibration_metrics,
    segmentation_metrics,
)
from aereo_water.evaluation.statistics import (
    add_performance_slices,
    paired_bootstrap_difference,
    summarize_slices,
    wilcoxon_paired,
)
from aereo_water.evaluation.thresholds import (
    calibrate_thresholds_from_probabilities,
    collect_original_resolution_probabilities,
)
from aereo_water.inference.predictor import SegFormerPredictor
from aereo_water.models.segformer import (
    SegFormerSpec,
    build_segformer,
    load_segformer_checkpoint,
    model_parameter_summary,
)
from aereo_water.pipeline.budget import estimate_compute_budget
from aereo_water.pipeline.compliance import (
    build_compliance_table,
    validate_inference_evidence,
)
from aereo_water.pipeline.failures import append_failure
from aereo_water.pipeline.state import STAGE_ORDER, StageState
from aereo_water.registry import (
    build_model_record,
    write_model_registry,
)
from aereo_water.training.engine import train_segformer
from aereo_water.training.hpo import (
    confirm_baseline_and_top_trials,
    config_from_parameters,
    run_hpo,
    run_seed_stability,
)
from aereo_water.utils import (
    available_disk_gb,
    ensure_minimum_disk,
    get_git_commit,
    json_dump,
    json_load,
    seed_everything,
    sha256_dataframe,
    sha256_file,
    utc_now_iso,
)

warnings.filterwarnings("once")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

In [ ]:
# Stable output layout.
DIRECTORIES = {
    name: OUTPUT_ROOT / name
    for name in [
        "registry",
        "figures",
        "tiling",
        "tracking",
        "hpo_smoke",
        "hpo",
        "confirmation",
        "stability",
        "final_training",
        "calibration",
        "evaluation",
        "statistics",
        "slices",
        "sample_predictions",
        "production_inference",
        "api_validation",
        "tests",
        "exports",
        "docs",
    ]
}
for directory in DIRECTORIES.values():
    directory.mkdir(parents=True, exist_ok=True)

ARTIFACT_SUFFIX = "" if RUN_PROFILE == "full" else "-smoke"

STAGE_STATE = StageState(OUTPUT_ROOT / "stage_state.json")
FAILURE_LEDGER = OUTPUT_ROOT / "failure_ledger.csv"
CONFIG_PATH = REPOSITORY_DIR / "configs" / "segformer_v3.yaml"
ACCEPTANCE_PATH = (
    REPOSITORY_DIR / "configs" / "acceptance_criteria.yaml"
)

CONFIG = load_config(CONFIG_PATH)
ACCEPTANCE_CRITERIA = yaml.safe_load(
    ACCEPTANCE_PATH.read_text(encoding="utf-8")
)
GIT_COMMIT = get_git_commit(REPOSITORY_DIR)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(CONFIG.training.seed)

def should_run(stage):
    if stage not in STAGE_ORDER:
        raise ValueError(stage)
    return not STAGE_STATE.is_complete(stage)

def require_artifact(path, description):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required {description} is missing: {path}. "
            "Run its prerequisite stage first or restore a resume bundle."
        )
    return path

print("Package version:", aereo_water.__version__)
print("Git commit:", GIT_COMMIT)
print("Device:", DEVICE)
print("Profile:", RUN_PROFILE)
print("Resume mode: automatic dependency traversal")

In [ ]:
# Persistent tracking locations. Database and artifact store are exported together.
MLFLOW_DATABASE = DIRECTORIES["tracking"] / "mlflow.db"
MLFLOW_TRACKING_URI = f"sqlite:///{MLFLOW_DATABASE}"
MLFLOW_ARTIFACT_ROOT = DIRECTORIES["tracking"] / "mlartifacts"
WANDB_ROOT = DIRECTORIES["tracking"] / "wandb"
WANDB_CACHE_ROOT = DIRECTORIES["tracking"] / "wandb_cache"

os.environ["WANDB_DIR"] = str(WANDB_ROOT)
os.environ["WANDB_CACHE_DIR"] = str(WANDB_CACHE_ROOT)

# Exact environment snapshot. pip_freeze.txt becomes the tested lock after success.
environment = {
    "timestamp_utc": utc_now_iso(),
    "run_profile": RUN_PROFILE,
    "run_stage": "automatic_resume",
    "repository_git_commit": GIT_COMMIT,
    "python": sys.version,
    "platform": platform.platform(),
    "pytorch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "package_version": aereo_water.__version__,
    "mlflow_tracking_uri": MLFLOW_TRACKING_URI,
    "mlflow_artifact_root": str(MLFLOW_ARTIFACT_ROOT),
    "wandb_mode": WANDB_MODE,
}
json_dump(environment, OUTPUT_ROOT / "run_environment.json")

freeze = run_command(
    [sys.executable, "-m", "pip", "freeze"],
    check=True,
)
(OUTPUT_ROOT / "pip_freeze.txt").write_text(
    freeze.stdout,
    encoding="utf-8",
)
display(pd.Series(environment, name="value").to_frame())

## 5. Compute-budget estimate and preflight safety checks

In [ ]:
# Apply profile-specific settings without mutating the repository YAML.
CONFIG.hpo.completed_trials = PROFILE["completed_hpo_trials"]
CONFIG.hpo.maximum_attempts = PROFILE["maximum_hpo_attempts"]
CONFIG.hpo.epochs_per_trial = PROFILE["hpo_epochs"]
CONFIG.hpo.train_subset_size = PROFILE["hpo_train_images"]
CONFIG.hpo.validation_subset_size = PROFILE["hpo_validation_images"]
CONFIG.hpo.confirmation_top_k = PROFILE["confirmation_top_k"]
CONFIG.hpo.confirmation_epochs = PROFILE["confirmation_epochs"]
CONFIG.hpo.stability_seeds = PROFILE["stability_seeds"]
CONFIG.training.epochs = PROFILE["final_epochs"]

budget = estimate_compute_budget(
    hpo_trials=CONFIG.hpo.completed_trials,
    hpo_epochs=CONFIG.hpo.epochs_per_trial,
    hpo_train_images=CONFIG.hpo.train_subset_size,
    confirmation_runs=1 + CONFIG.hpo.confirmation_top_k,
    confirmation_epochs=CONFIG.hpo.confirmation_epochs,
    stability_runs=len(CONFIG.hpo.stability_seeds),
    stability_epochs=PROFILE["stability_epochs"],
    full_train_images=CONFIG.data.train_count,
    final_epochs=CONFIG.training.epochs,
    batch_size=CONFIG.training.batch_size,
    gradient_accumulation_steps=(
        CONFIG.training.gradient_accumulation_steps
    ),
    minimum_free_disk_gb=PROFILE["minimum_free_disk_gb"],
)
free_disk_gb = available_disk_gb(OUTPUT_ROOT)
budget_frame = pd.Series(
    {**budget.to_dict(), "current_free_disk_gb": free_disk_gb},
    name="value",
).to_frame()
display(budget_frame)

ensure_minimum_disk(
    OUTPUT_ROOT,
    minimum_gb=budget.minimum_free_disk_gb,
)
if RUN_PROFILE == "full" and not torch.cuda.is_available():
    raise RuntimeError(
        "The full profile requires a GPU. Use the Kaggle GPU accelerator."
    )

## 6. Dataset discovery and validation

The discovery logic does not assume a fixed Kaggle dataset slug. It searches for candidate
`Images` and `Masks` directories, chooses the pair with the largest supported-file counts, and
then validates every pair.

Validation includes:

- one-to-one stem pairing;
- duplicate stem rejection;
- decoding;
- matching dimensions;
- binary-mask interpretation;
- water fraction;
- exact SHA-256 hashes;
- perceptual image hashes;
- error-table export.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
SUPPORTED = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

def supported_count(directory):
    return sum(
        path.is_file() and path.suffix.lower() in SUPPORTED
        for path in directory.rglob("*")
    )

# Select Images and Masks from the same dataset root. Choosing the largest
# independent Images and Masks folders could silently pair different datasets.
paired_dataset_candidates = []
for directory in INPUT_ROOT.rglob("*"):
    if not directory.is_dir():
        continue
    children = {
        child.name.lower(): child
        for child in directory.iterdir()
        if child.is_dir()
    }
    if "images" not in children or "masks" not in children:
        continue
    images = children["images"]
    masks = children["masks"]
    image_count = supported_count(images)
    mask_count = supported_count(masks)
    paired_dataset_candidates.append(
        {
            "root": directory,
            "images": images,
            "masks": masks,
            "image_count": image_count,
            "mask_count": mask_count,
            "paired_capacity": min(image_count, mask_count),
        }
    )

if not paired_dataset_candidates:
    raise FileNotFoundError(
        "Attach the Kaggle Satellite Images of Water Bodies dataset. "
        "No common parent containing Images/ and Masks/ was found."
    )

dataset_choice = max(
    paired_dataset_candidates,
    key=lambda item: (
        item["paired_capacity"],
        item["image_count"] + item["mask_count"],
    ),
)
IMAGE_DIR = dataset_choice["images"]
MASK_DIR = dataset_choice["masks"]
DATASET_ROOT = dataset_choice["root"]

print("Dataset root:", DATASET_ROOT)
print("Images:", IMAGE_DIR, dataset_choice["image_count"])
print("Masks:", MASK_DIR, dataset_choice["mask_count"])
display(pd.DataFrame(paired_dataset_candidates).sort_values(
    ["paired_capacity", "image_count"],
    ascending=False,
).head(10))

In [ ]:
if should_run("data"):
    STAGE_STATE.start("data")
    try:
        raw_pairs = discover_pairs(IMAGE_DIR, MASK_DIR)
        validated_manifest, data_errors = validate_manifest(
            raw_pairs,
            compute_sha256=True,
            compute_perceptual_hash=True,
        )
        data_errors.to_csv(
            DIRECTORIES["registry"] / "data_errors.csv",
            index=False,
        )
        if not data_errors.empty:
            raise RuntimeError(
                f"{len(data_errors)} invalid pairs were found. "
                "Inspect registry/data_errors.csv before training."
            )
        if len(validated_manifest) != 2841:
            raise RuntimeError(
                f"Expected 2,841 valid pairs, found {len(validated_manifest)}."
            )
    except Exception as exc:
        STAGE_STATE.fail("data", str(exc))
        append_failure(
            FAILURE_LEDGER,
            stage="data",
            error=exc,
            root_cause="Dataset contract violation",
        )
        raise
else:
    validated_path = require_artifact(
        DIRECTORIES["registry"] / "validated_manifest.csv",
        "validated manifest",
    )
    validated_manifest = pd.read_csv(validated_path)

validated_manifest.to_csv(
    DIRECTORIES["registry"] / "validated_manifest.csv",
    index=False,
)
display(validated_manifest.head())

## 7. Exact split reconstruction, portable registry, and leakage audit

In [ ]:
def locate_historical_segformer_results():
    filename = "experiment_C_segformer_all_2841.csv"
    candidates = [
        REPOSITORY_DIR / "results" / "full" / filename,
        REPOSITORY_DIR / "results" / filename,
    ]
    candidates.extend(INPUT_ROOT.rglob(filename))
    return next((path for path in candidates if path.exists()), None)

HISTORICAL_SEGFORMER_CSV = locate_historical_segformer_results()
print("Historical split source:", HISTORICAL_SEGFORMER_CSV)

In [ ]:
if should_run("data"):
    if HISTORICAL_SEGFORMER_CSV is not None:
        manifest = recover_historical_split(
            validated_manifest,
            HISTORICAL_SEGFORMER_CSV,
        )
        split_origin = "recovered from historical Experiment C registry"
    else:
        manifest = assign_exact_split(
            validated_manifest,
            train_count=CONFIG.data.train_count,
            validation_count=CONFIG.data.validation_count,
            test_count=CONFIG.data.test_count,
            seed=CONFIG.data.split_seed,
        )
        split_origin = "deterministic stratified fallback"

    assert_split_integrity(manifest)
    counts = manifest["split"].value_counts().to_dict()
    expected_counts = {
        "train": 1991,
        "validation": 429,
        "test": 421,
    }
    if counts != expected_counts:
        raise AssertionError(f"{counts} != {expected_counts}")

    runtime_manifest_path = (
        DIRECTORIES["registry"] / "runtime_manifest.csv"
    )
    manifest.to_csv(runtime_manifest_path, index=False)

    portable_manifest = make_portable_registry(
        manifest,
        dataset_root=DATASET_ROOT,
    )
    (
        SPLIT_REGISTRY_PATH,
        DATA_REGISTRY_PATH,
        SPLIT_REGISTRY_SHA256,
    ) = write_data_registry(
        portable_manifest,
        output_csv=DIRECTORIES["registry"] / "split_registry.csv",
        output_json=DIRECTORIES["registry"] / "data_registry.json",
        dataset_name="Satellite Images of Water Bodies",
        dataset_source="Kaggle",
        split_seed=CONFIG.data.split_seed,
        git_commit=GIT_COMMIT,
        duplicate_policy=(
            "exact SHA-256 rejection plus perceptual-hash audit; "
            "historical split retained for comparison"
        ),
        split_before_tiling=True,
    )
    registry_metadata = json_load(DATA_REGISTRY_PATH)
    registry_metadata["split_origin"] = split_origin
    json_dump(registry_metadata, DATA_REGISTRY_PATH)
else:
    # Rebuild runtime paths from the portable registry instead of trusting
    # Kaggle absolute paths saved by a previous session.
    SPLIT_REGISTRY_PATH = require_artifact(
        DIRECTORIES["registry"] / "split_registry.csv",
        "split registry",
    )
    DATA_REGISTRY_PATH = require_artifact(
        DIRECTORIES["registry"] / "data_registry.json",
        "data registry",
    )
    portable_manifest = pd.read_csv(SPLIT_REGISTRY_PATH)
    manifest = portable_manifest.copy()
    manifest["image_path"] = manifest["image_relative_path"].map(
        lambda relative: str(DATASET_ROOT / Path(relative))
    )
    manifest["mask_path"] = manifest["mask_relative_path"].map(
        lambda relative: str(DATASET_ROOT / Path(relative))
    )
    missing_runtime_files = manifest.loc[
        ~manifest["image_path"].map(lambda value: Path(value).exists())
        | ~manifest["mask_path"].map(lambda value: Path(value).exists()),
        "image_id",
    ]
    if not missing_runtime_files.empty:
        raise FileNotFoundError(
            "Portable registry path reconstruction failed for IDs: "
            f"{missing_runtime_files.head(20).tolist()}"
        )
    SPLIT_REGISTRY_SHA256 = json_load(
        DATA_REGISTRY_PATH
    )["split_registry_sha256"]

display(manifest.groupby("split").agg(
    images=("image_id", "size"),
    mean_water_fraction=("water_fraction", "mean"),
    empty_masks=("water_fraction", lambda values: int((values == 0).sum())),
))
print("Split registry SHA-256:", SPLIT_REGISTRY_SHA256)

In [ ]:
if should_run("data") and RUN_NEAR_DUPLICATE_AUDIT:
    # Complete perceptual-hash comparison across every split boundary.
    # The historical split is retained for comparability; suspected pairs are
    # surfaced as audit evidence rather than silently reassigned.
    near_duplicates = near_duplicate_audit(
        manifest,
        hamming_threshold=(
            CONFIG.data.near_duplicate_hamming_threshold
        ),
        maximum_pairs=None,
    )
    near_duplicates.to_csv(
        DIRECTORIES["registry"] / "near_duplicate_audit.csv",
        index=False,
    )
    near_duplicate_metadata = dict(near_duplicates.attrs)
    near_duplicate_metadata[
        "suspected_cross_split_pairs"
    ] = int(len(near_duplicates))
    json_dump(
        near_duplicate_metadata,
        DIRECTORIES["registry"]
        / "near_duplicate_audit_metadata.json",
    )
else:
    audit_path = DIRECTORIES["registry"] / "near_duplicate_audit.csv"
    near_duplicates = (
        pd.read_csv(audit_path)
        if audit_path.exists()
        else pd.DataFrame()
    )
    metadata_path = (
        DIRECTORIES["registry"]
        / "near_duplicate_audit_metadata.json"
    )
    near_duplicate_metadata = (
        json_load(metadata_path)
        if metadata_path.exists()
        else {
            "audit_complete": False,
            "reason": "Near-duplicate audit was disabled.",
        }
    )

print("Potential cross-split near-duplicate pairs:", len(near_duplicates))
display(pd.Series(near_duplicate_metadata, name="value").to_frame())
display(near_duplicates.head(20))

In [ ]:
# Materialize only train and validation for model selection.
train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
validation_df = manifest[
    manifest["split"] == "validation"
].reset_index(drop=True)

def assert_no_test_rows(*frames):
    for frame in frames:
        labels = set(frame["split"].astype(str).str.lower())
        if "test" in labels:
            raise RuntimeError(
                "Test rows entered a model-selection dataframe."
            )

assert_no_test_rows(train_df, validation_df)
print("Model-selection rows:", len(train_df), len(validation_df))

## 8. Exploratory data analysis tied to modeling risk

In [ ]:
FIGURE_DIR = DIRECTORIES["figures"]

eda_summary = pd.DataFrame(
    {
        "metric": [
            "pairs",
            "minimum width",
            "maximum width",
            "minimum height",
            "maximum height",
            "median water fraction",
            "empty masks",
            "images above 75% water",
        ],
        "value": [
            len(manifest),
            int(manifest["width"].min()),
            int(manifest["width"].max()),
            int(manifest["height"].min()),
            int(manifest["height"].max()),
            float(manifest["water_fraction"].median()),
            int((manifest["water_fraction"] == 0).sum()),
            int((manifest["water_fraction"] >= 0.75).sum()),
        ],
    }
)
eda_summary.to_csv(
    DIRECTORIES["registry"] / "eda_summary.csv",
    index=False,
)
display(eda_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for split, group in manifest.groupby("split"):
    axes[0].hist(
        group["water_fraction"],
        bins=30,
        alpha=0.45,
        label=split,
    )
axes[0].set_title("Water-pixel fraction by split")
axes[0].set_xlabel("Water fraction")
axes[0].set_ylabel("Images")
axes[0].legend()

axes[1].scatter(
    manifest["width"],
    manifest["height"],
    c=manifest["water_fraction"],
    s=12,
)
axes[1].set_title("Image dimensions")
axes[1].set_xlabel("Width")
axes[1].set_ylabel("Height")

manifest.boxplot(
    column="water_fraction",
    by="split",
    ax=axes[2],
)
axes[2].set_title("Split water-coverage balance")
axes[2].set_xlabel("Split")
axes[2].set_ylabel("Water fraction")
fig.suptitle("")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "dataset_eda.png", dpi=200)
plt.show()

In [ ]:
# Display examples across the water-coverage distribution instead of only
# visually convenient examples.
quantiles = [0.0, 0.1, 0.25, 0.5, 0.75, 0.95]
sample_rows = []
for quantile in quantiles:
    target_fraction = manifest["water_fraction"].quantile(quantile)
    index = (
        manifest["water_fraction"] - target_fraction
    ).abs().idxmin()
    sample_rows.append(manifest.loc[index])

fig, axes = plt.subplots(len(sample_rows), 2, figsize=(10, 4 * len(sample_rows)))
for row_axes, row in zip(axes, sample_rows):
    with Image.open(row["image_path"]) as raw:
        image = raw.convert("RGB")
    with Image.open(row["mask_path"]) as raw:
        mask = raw.convert("L")
    row_axes[0].imshow(image)
    row_axes[0].set_title(
        f"{row['image_id']} | water={row['water_fraction']:.3f}"
    )
    row_axes[1].imshow(mask, cmap="gray")
    row_axes[1].set_title("Ground-truth mask")
    for axis in row_axes:
        axis.axis("off")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "dataset_examples_by_water_fraction.png", dpi=180)
plt.show()

## 9. Aspect-ratio-safe preprocessing, normalization, and synchronized augmentation

In [ ]:
# Load only the processor for preprocessing inspection.
MODEL_SPEC = SegFormerSpec(
    model_id=CONFIG.model.model_id,
    num_labels=CONFIG.model.num_labels,
    id2label=CONFIG.model.id2label,
    label2id=CONFIG.model.label2id,
)
_, PREPROCESSOR = build_segformer(MODEL_SPEC)

sample = train_df.sample(n=1, random_state=CONFIG.training.seed).iloc[0]
with Image.open(sample["image_path"]) as raw:
    original_image = raw.convert("RGB")
with Image.open(sample["mask_path"]) as raw:
    original_mask = Image.fromarray(
        (np.asarray(raw.convert("L")) > 0).astype(np.uint8)
    )

stretch_image, stretch_mask, stretch_transform = resize_pair(
    original_image,
    original_mask,
    size=CONFIG.data.image_size,
    policy="stretch",
)
letterbox_image, letterbox_mask, letterbox_transform = resize_pair(
    original_image,
    original_mask,
    size=CONFIG.data.image_size,
    policy="letterbox",
)

training_dataset = SegmentationTrainingDataset(
    train_df.iloc[[0]],
    PREPROCESSOR,
    image_size=CONFIG.data.image_size,
    resize_policy="letterbox",
    augmentation_profile="moderate",
    base_seed=CONFIG.training.seed,
)
training_dataset.set_epoch(1)
augmented = training_dataset[0]

print("Original size:", original_image.size)
print("Stretch transform:", stretch_transform)
print("Letterbox transform:", letterbox_transform)
print("Normalized tensor shape:", tuple(augmented["pixel_values"].shape))
print(
    "Normalized tensor min/mean/max:",
    float(augmented["pixel_values"].min()),
    float(augmented["pixel_values"].mean()),
    float(augmented["pixel_values"].max()),
)
print("Mask values:", torch.unique(augmented["labels"]).tolist())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes[0, 0].imshow(original_image)
axes[0, 0].set_title(f"Original {original_image.size}")
axes[0, 1].imshow(stretch_image)
axes[0, 1].set_title("Stretch resize — comparison only")
axes[0, 2].imshow(letterbox_image)
axes[0, 2].set_title("Letterbox resize — maintained policy")

axes[1, 0].imshow(original_mask, cmap="gray")
axes[1, 0].set_title("Original mask")
axes[1, 1].imshow(stretch_mask, cmap="gray")
axes[1, 1].set_title("Stretch mask")
training_label_view = augmented["labels"].numpy().astype(float)
training_label_view[training_label_view == 255] = np.nan
axes[1, 2].imshow(training_label_view, cmap="gray", vmin=0, vmax=1)
axes[1, 2].set_title("Augmented mask; padded pixels ignored")

for axis in axes.ravel():
    axis.axis("off")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "preprocessing_and_augmentation.png",
    dpi=200,
)
plt.show()

### Preprocessing decision

The maintained V3 policy uses **aspect-ratio-preserving letterbox resize** followed by removal of
padding and restoration to original dimensions during validation and inference. Artificial
letterbox pixels use label `255` and are excluded from both cross-entropy and Dice loss; they are
not treated as genuine non-water background. The stretch policy is shown only as an explicit
comparison and is not silently mixed into the final experiment.


## 10. Overlapping tiling and metadata-preserving GeoTIFF reconstruction

In [ ]:
if should_run("data"):
    # Choose the largest available source and guarantee a multi-tile mosaic.
    largest = manifest.assign(
        pixel_count=manifest["width"] * manifest["height"]
    ).sort_values("pixel_count", ascending=False).iloc[0]
    with Image.open(largest["image_path"]) as raw:
        source_rgb = np.asarray(raw.convert("RGB"))

    demo_mosaic = build_multitile_demo(
        source_rgb,
        tile_size=CONFIG.data.tile_size,
        overlap=CONFIG.data.tile_overlap,
    )
    demo_tiles, tile_manifest = tile_array(
        demo_mosaic,
        tile_size=CONFIG.data.tile_size,
        overlap=CONFIG.data.tile_overlap,
        parent_image_id=str(largest["image_id"]),
    )
    reconstructed = reconstruct_tiles(
        demo_tiles,
        tile_manifest,
        output_height=demo_mosaic.shape[0],
        output_width=demo_mosaic.shape[1],
    )

    assert len(demo_tiles) > 1
    assert tile_manifest["row_offset"].nunique() > 1
    assert tile_manifest["column_offset"].nunique() > 1
    assert np.array_equal(reconstructed, demo_mosaic)

    tile_manifest.to_csv(
        DIRECTORIES["tiling"] / "tiling_manifest.csv",
        index=False,
    )
    Image.fromarray(demo_mosaic).save(
        DIRECTORIES["tiling"] / "source_mosaic.png"
    )
    Image.fromarray(reconstructed).save(
        DIRECTORIES["tiling"] / "reconstructed_mosaic.png"
    )

    print("Mosaic shape:", demo_mosaic.shape)
    print("Overlapping tiles:", len(demo_tiles))
    print(
        "Maximum reconstruction error:",
        int(np.abs(
            reconstructed.astype(np.int64)
            - demo_mosaic.astype(np.int64)
        ).max()),
    )
else:
    tile_manifest = pd.read_csv(
        require_artifact(
            DIRECTORIES["tiling"] / "tiling_manifest.csv",
            "tiling manifest",
        )
    )
    with Image.open(
        require_artifact(
            DIRECTORIES["tiling"] / "source_mosaic.png",
            "tiling source mosaic",
        )
    ) as raw:
        demo_mosaic = np.asarray(raw.convert("RGB"))
    with Image.open(
        require_artifact(
            DIRECTORIES["tiling"] / "reconstructed_mosaic.png",
            "tiling reconstruction",
        )
    ) as raw:
        reconstructed = np.asarray(raw.convert("RGB"))
display(tile_manifest.head())

In [ ]:
# Geospatial capability test:
# If a truly georeferenced source exists, use it. Otherwise create an explicitly
# labelled synthetic GeoTIFF from a real RGB sample and verify that CRS,
# affine transform, bounds, nodata, and pixels survive tiling/reconstruction.
import rasterio
from rasterio.transform import from_origin

geotiff_candidates = []
for path in list(IMAGE_DIR.rglob("*.tif")) + list(IMAGE_DIR.rglob("*.tiff")):
    try:
        with rasterio.open(path) as source:
            if source.crs is not None:
                geotiff_candidates.append(path)
    except Exception:
        continue

if should_run("data"):
    if geotiff_candidates:
        geotiff_source = geotiff_candidates[0]
        geotiff_origin = "dataset georeferenced raster"
    else:
        geotiff_source = DIRECTORIES["tiling"] / "synthetic_georeferenced_demo.tif"
        synthetic = demo_mosaic[:1600, :1800]
        channels = np.moveaxis(synthetic, -1, 0)
        with rasterio.open(
            geotiff_source,
            "w",
            driver="GTiff",
            width=synthetic.shape[1],
            height=synthetic.shape[0],
            count=channels.shape[0],
            dtype=channels.dtype,
            crs="EPSG:4326",
            transform=from_origin(77.0, 29.0, 0.0001, 0.0001),
            nodata=0,
        ) as destination:
            destination.write(channels)
        geotiff_origin = (
            "synthetic geospatial metadata applied to a real dataset image "
            "for capability validation"
        )

    geotiff_manifest = tile_geotiff(
        geotiff_source,
        DIRECTORIES["tiling"] / "geotiff_tiles",
        tile_size=CONFIG.data.tile_size,
        overlap=CONFIG.data.tile_overlap,
    )
    reconstructed_geotiff = reconstruct_geotiff(
        geotiff_manifest,
        DIRECTORIES["tiling"] / "reconstructed_geotiff.tif",
        source_reference_path=geotiff_source,
    )
    geotiff_manifest["demonstration_origin"] = geotiff_origin
    geotiff_manifest.to_csv(
        DIRECTORIES["tiling"] / "geotiff_tiling_manifest.csv",
        index=False,
    )

    with rasterio.open(geotiff_source) as source, rasterio.open(
        reconstructed_geotiff
    ) as restored:
        assert source.crs == restored.crs
        assert source.transform == restored.transform
        assert source.bounds == restored.bounds
        assert np.array_equal(source.read(), restored.read())
        print("GeoTIFF metadata and pixel parity passed.")
        print("Demonstration origin:", geotiff_origin)
else:
    geotiff_manifest = pd.read_csv(
        require_artifact(
            DIRECTORIES["tiling"] / "geotiff_tiling_manifest.csv",
            "GeoTIFF tiling manifest",
        )
    )
display(geotiff_manifest.head())

if should_run("data"):
    STAGE_STATE.complete(
        "data",
        evidence=[
            str(DIRECTORIES["registry"] / "runtime_manifest.csv"),
            str(SPLIT_REGISTRY_PATH),
            str(DATA_REGISTRY_PATH),
            str(
                DIRECTORIES["registry"]
                / "near_duplicate_audit_metadata.json"
            ),
            str(DIRECTORIES["tiling"] / "tiling_manifest.csv"),
            str(
                DIRECTORIES["tiling"]
                / "geotiff_tiling_manifest.csv"
            ),
        ],
    )


In [ ]:
# Visualize the overlapping grid over the source mosaic.
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(demo_mosaic)
for row in tile_manifest.itertuples(index=False):
    rectangle = plt.Rectangle(
        (row.column_offset, row.row_offset),
        row.valid_width,
        row.valid_height,
        fill=False,
        linewidth=1,
    )
    axes[0].add_patch(rectangle)
axes[0].set_title(
    f"Overlapping grid: {len(tile_manifest)} tiles, "
    f"{CONFIG.data.tile_overlap}px overlap"
)
axes[1].imshow(reconstructed)
axes[1].set_title("Exact overlap-averaged reconstruction")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "tiling_reconstruction.png", dpi=200)
plt.show()

## 11. SegFormer architecture, same-code control, and tracking design

In [ ]:
# Rebuild a clean model for architecture inspection and immediately release it
# before the training stages.
architecture_model, architecture_processor = build_segformer(MODEL_SPEC)
parameter_summary = model_parameter_summary(architecture_model)
architecture_config = {
    "base_model": MODEL_SPEC.model_id,
    "classes": MODEL_SPEC.resolved_id2label(),
    "input_policy": (
        f"{CONFIG.data.resize_policy} to "
        f"{CONFIG.data.image_size}x{CONFIG.data.image_size}"
    ),
    "training_loss": (
        f"{CONFIG.training.ce_weight:.1f} Cross-Entropy + "
        f"{CONFIG.training.dice_weight:.1f} Dice"
    ),
    "model_selection_metric": (
        "mean per-image original-resolution validation IoU at threshold 0.50"
    ),
    **parameter_summary,
}
json_dump(
    architecture_config,
    OUTPUT_ROOT / "model_architecture.json",
)
display(pd.Series(architecture_config, name="value").to_frame())

del architecture_model, architecture_processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### SegFormer rationale

SegFormer combines a hierarchical transformer encoder with a lightweight all-MLP decoder. It is
well suited to this assignment because it supports fully automatic dense prediction, captures
multi-scale context, and has a substantially simpler inference path than a prompted SAM system.

### Same-code control

The documented historical hyperparameter configuration is not treated as merely an Optuna
candidate. It is always rerun under the V3 implementation during confirmation. This is a
same-engine control: it matches the documented optimizer/loss/batch settings while intentionally
using the corrected V3 preprocessing, validation objective, and training implementation:

```text
learning rate  = 6e-5
weight decay   = 1e-4
CE weight      = 0.6
Dice weight    = 0.4
warm-up ratio  = 0.1
augmentation   = moderate
batch size       = 8
```

This separates **HPO gains** from gains caused by rewritten preprocessing or training code.

In [ ]:
BASELINE_PARAMETERS = {
    "learning_rate": 6e-5,
    "weight_decay": 1e-4,
    "ce_weight": 0.6,
    "dice_weight": 0.4,
    "warmup_ratio": 0.1,
    "augmentation_profile": "moderate",
    "batch_size": 8,
    "gradient_accumulation_steps": 1,
}

tracking_design = pd.DataFrame(
    [
        {
            "system": "MLflow",
            "role": "authoritative system of record",
            "storage": str(MLFLOW_DATABASE),
            "artifacts": str(MLFLOW_ARTIFACT_ROOT),
        },
        {
            "system": "Weights & Biases",
            "role": "optional visualization mirror",
            "storage": str(WANDB_ROOT),
            "artifacts": "offline runs and W&B artifacts",
        },
        {
            "system": "Optuna",
            "role": "search, pruning, and trial ranking",
            "storage": str(DIRECTORIES["hpo"] / "optuna.db"),
            "artifacts": "trial table, fingerprint, best-trial JSON",
        },
    ]
)
display(tracking_design)

## 12. Pipeline smoke test

In [ ]:
# The smoke test validates the entire training/tracking path on a tiny subset.
# Its metrics are not scientific results and are excluded from the final report.
SMOKE_HPO_CONFIG = HPOConfig(
    completed_trials=1,
    maximum_attempts=2,
    epochs_per_trial=1,
    train_subset_size=64,
    validation_subset_size=16,
    confirmation_top_k=1,
    confirmation_epochs=1,
    stability_seeds=[42],
    study_name="water_segformer_smoke",
)

if not (
    DIRECTORIES["hpo_smoke"] / "best_trial.json"
).exists():
    smoke_study, smoke_trials = run_hpo(
        train_df,
        validation_df,
        model_spec=MODEL_SPEC,
        base_training_config=CONFIG.training,
        hpo_config=SMOKE_HPO_CONFIG,
        image_size=CONFIG.data.image_size,
        resize_policy=CONFIG.data.resize_policy,
        output_dir=DIRECTORIES["hpo_smoke"],
        device=DEVICE,
        mlflow_tracking_uri=MLFLOW_TRACKING_URI,
        mlflow_artifact_root=MLFLOW_ARTIFACT_ROOT,
        wandb_project=CONFIG.tracking.wandb_project,
        wandb_mode=WANDB_MODE,
        wandb_root=WANDB_ROOT,
        repo_git_commit=GIT_COMMIT,
        split_registry_sha256=SPLIT_REGISTRY_SHA256,
        baseline_parameters=BASELINE_PARAMETERS,
    )
    display(smoke_trials)
else:
    print("Smoke test skipped or already completed.")

## 13. Fingerprint-aware Optuna search

In [ ]:
if should_run("hpo"):
    STAGE_STATE.require("hpo")
    STAGE_STATE.start("hpo")
    try:
        study, hpo_trials = run_hpo(
            train_df,
            validation_df,
            model_spec=MODEL_SPEC,
            base_training_config=CONFIG.training,
            hpo_config=CONFIG.hpo,
            image_size=CONFIG.data.image_size,
            resize_policy=CONFIG.data.resize_policy,
            output_dir=DIRECTORIES["hpo"],
            device=DEVICE,
            mlflow_tracking_uri=MLFLOW_TRACKING_URI,
            mlflow_artifact_root=MLFLOW_ARTIFACT_ROOT,
            wandb_project=CONFIG.tracking.wandb_project,
            wandb_mode=WANDB_MODE,
            wandb_root=WANDB_ROOT,
            repo_git_commit=GIT_COMMIT,
            split_registry_sha256=SPLIT_REGISTRY_SHA256,
            baseline_parameters=BASELINE_PARAMETERS,
        )
        STAGE_STATE.complete(
            "hpo",
            evidence=[
                str(DIRECTORIES["hpo"] / "optuna.db"),
                str(DIRECTORIES["hpo"] / "trials.csv"),
                str(DIRECTORIES["hpo"] / "best_trial.json"),
            ],
        )
    except Exception as exc:
        STAGE_STATE.fail("hpo", str(exc))
        raise
else:
    import optuna
    hpo_db = require_artifact(
        DIRECTORIES["hpo"] / "optuna.db",
        "Optuna study database",
    )
    study = optuna.load_study(
        study_name=CONFIG.hpo.study_name,
        storage=f"sqlite:///{hpo_db}",
    )
    hpo_trials = pd.read_csv(
        require_artifact(
            DIRECTORIES["hpo"] / "trials.csv",
            "HPO trial table",
        )
    )

display(
    hpo_trials.sort_values(
        "value",
        ascending=False,
        na_position="last",
    ).head(20)
)

In [ ]:
# Report completed, pruned, and failed trials separately.
trial_state_summary = (
    hpo_trials["state"]
    .astype(str)
    .value_counts()
    .rename_axis("state")
    .reset_index(name="trials")
)
display(trial_state_summary)

complete_trials = hpo_trials[
    hpo_trials["state"].astype(str).str.contains("COMPLETE")
].copy()
if len(complete_trials) < CONFIG.hpo.completed_trials:
    raise RuntimeError(
        f"Expected {CONFIG.hpo.completed_trials} completed trials, "
        f"found {len(complete_trials)}."
    )

plt.figure(figsize=(10, 5))
plt.plot(
    complete_trials["number"],
    complete_trials["value"],
    marker="o",
)
plt.xlabel("Optuna trial")
plt.ylabel("Best original-resolution validation IoU")
plt.title("HPO optimization history")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "hpo_optimization_history.png", dpi=200)
plt.show()

## 14. Same-code baseline and top-candidate confirmation

In [ ]:
if should_run("confirmation"):
    STAGE_STATE.require("confirmation")
    STAGE_STATE.start("confirmation")
    try:
        confirmation = confirm_baseline_and_top_trials(
            study,
            train_df,
            validation_df,
            model_spec=MODEL_SPEC,
            base_training_config=CONFIG.training,
            baseline_parameters=BASELINE_PARAMETERS,
            top_k=CONFIG.hpo.confirmation_top_k,
            confirmation_epochs=CONFIG.hpo.confirmation_epochs,
            image_size=CONFIG.data.image_size,
            resize_policy=CONFIG.data.resize_policy,
            output_dir=DIRECTORIES["confirmation"],
            device=DEVICE,
            mlflow_tracking_uri=MLFLOW_TRACKING_URI,
            mlflow_artifact_root=MLFLOW_ARTIFACT_ROOT,
            wandb_project=CONFIG.tracking.wandb_project,
            wandb_mode=WANDB_MODE,
            wandb_root=WANDB_ROOT,
            repo_git_commit=GIT_COMMIT,
            split_registry_sha256=SPLIT_REGISTRY_SHA256,
        )
        STAGE_STATE.complete(
            "confirmation",
            evidence=[
                str(
                    DIRECTORIES["confirmation"]
                    / "confirmation_results.csv"
                )
            ],
        )
    except Exception as exc:
        STAGE_STATE.fail("confirmation", str(exc))
        raise
else:
    confirmation = pd.read_csv(
        require_artifact(
            DIRECTORIES["confirmation"] / "confirmation_results.csv",
            "confirmation results",
        )
    )

display(confirmation)

In [ ]:
# Selection is validation-only. Parameters are stored as canonical JSON.
selected_confirmation = confirmation.sort_values(
    ["best_validation_iou", "best_validation_dice"],
    ascending=False,
).iloc[0]
SELECTED_PARAMETERS = json.loads(
    selected_confirmation["parameters_json"]
)
SELECTED_PARAMETERS_PATH = (
    OUTPUT_ROOT / "selected_final_parameters.json"
)
json_dump(SELECTED_PARAMETERS, SELECTED_PARAMETERS_PATH)

baseline_confirmation = confirmation[
    confirmation["label"] == "same_code_historical_baseline"
]
selection_summary = {
    "selected_label": selected_confirmation["label"],
    "selected_validation_iou": float(
        selected_confirmation["best_validation_iou"]
    ),
    "selected_validation_dice": float(
        selected_confirmation["best_validation_dice"]
    ),
    "same_code_baseline_validation_iou": (
        float(baseline_confirmation.iloc[0]["best_validation_iou"])
        if len(baseline_confirmation)
        else None
    ),
    "selection_test_split_used": False,
    "selected_parameters": SELECTED_PARAMETERS,
}
json_dump(
    selection_summary,
    DIRECTORIES["confirmation"] / "selection_summary.json",
)
display(pd.Series(selection_summary, name="value").to_frame())

## 15. Random-seed stability

In [ ]:
if RUN_SEED_STABILITY:
    if should_run("stability"):
        STAGE_STATE.start("stability")
        try:
            stability = run_seed_stability(
                train_df,
                validation_df,
                selected_parameters=SELECTED_PARAMETERS,
                seeds=CONFIG.hpo.stability_seeds,
                epochs=PROFILE["stability_epochs"],
                model_spec=MODEL_SPEC,
                base_training_config=CONFIG.training,
                image_size=CONFIG.data.image_size,
                resize_policy=CONFIG.data.resize_policy,
                output_dir=DIRECTORIES["stability"],
                device=DEVICE,
                mlflow_tracking_uri=MLFLOW_TRACKING_URI,
                mlflow_artifact_root=MLFLOW_ARTIFACT_ROOT,
                wandb_project=CONFIG.tracking.wandb_project,
                wandb_mode=WANDB_MODE,
                wandb_root=WANDB_ROOT,
                repo_git_commit=GIT_COMMIT,
                split_registry_sha256=SPLIT_REGISTRY_SHA256,
            )
            STAGE_STATE.complete(
                "stability",
                evidence=[
                    str(
                        DIRECTORIES["stability"]
                        / "seed_stability.csv"
                    )
                ],
            )
        except Exception as exc:
            STAGE_STATE.fail("stability", str(exc))
            raise
    else:
        stability = pd.read_csv(
            require_artifact(
                DIRECTORIES["stability"] / "seed_stability.csv",
                "seed-stability results",
            )
        )
    display(stability)
    print(
        "Mean ± SD validation IoU:",
        stability["best_validation_iou"].mean(),
        "±",
        stability["best_validation_iou"].std(ddof=1)
        if len(stability) > 1
        else 0.0,
    )
else:
    stability = pd.DataFrame()
    print("Seed stability disabled by user control.")

The final production seed remains the pre-declared seed `42`. Stability analysis quantifies
sensitivity; it does not authorize choosing a seed using held-out test performance.

## 16. Resumable final full training

In [ ]:
FINAL_TRAINING_DIR = DIRECTORIES["final_training"]
FINAL_CHECKPOINT = FINAL_TRAINING_DIR / "best_checkpoint"
FINAL_STATE = FINAL_TRAINING_DIR / "last_state.pt"

if should_run("final_train"):
    STAGE_STATE.require("final_train")
    STAGE_STATE.start("final_train")
    try:
        final_config = config_from_parameters(
            CONFIG.training,
            SELECTED_PARAMETERS,
            epochs=PROFILE["final_epochs"],
            seed=CONFIG.training.seed,
            save_every_epoch=True,
        )
        resume_from = FINAL_STATE if FINAL_STATE.exists() else None
        final_metadata = train_segformer(
            train_df,
            validation_df,
            model_spec=MODEL_SPEC,
            config=final_config,
            image_size=CONFIG.data.image_size,
            resize_policy=CONFIG.data.resize_policy,
            output_dir=FINAL_TRAINING_DIR,
            device=DEVICE,
            run_name="segformer_v3_final_training",
            experiment_name=CONFIG.tracking.mlflow_experiment_final,
            mlflow_tracking_uri=MLFLOW_TRACKING_URI,
            mlflow_artifact_root=MLFLOW_ARTIFACT_ROOT,
            wandb_project=CONFIG.tracking.wandb_project,
            wandb_mode=WANDB_MODE,
            wandb_root=WANDB_ROOT,
            repo_git_commit=GIT_COMMIT,
            split_registry_sha256=SPLIT_REGISTRY_SHA256,
            resume_from=resume_from,
        )
        STAGE_STATE.complete(
            "final_train",
            evidence=[
                str(FINAL_CHECKPOINT),
                str(FINAL_STATE),
                str(FINAL_TRAINING_DIR / "history.csv"),
            ],
        )
    except Exception as exc:
        STAGE_STATE.fail("final_train", str(exc))
        append_failure(
            FAILURE_LEDGER,
            stage="final_train",
            error=exc,
            configuration=SELECTED_PARAMETERS,
        )
        raise
else:
    final_metadata = json_load(
        require_artifact(
            FINAL_TRAINING_DIR / "training_metadata.json",
            "final training metadata",
        )
    )

require_artifact(FINAL_CHECKPOINT, "final best checkpoint")
final_history = pd.read_csv(
    require_artifact(
        FINAL_TRAINING_DIR / "history.csv",
        "final training history",
    )
)
display(final_history)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(final_history["epoch"], final_history["train_loss"], marker="o")
axes[0].set_title("Training loss")
axes[0].set_xlabel("Epoch")

axes[1].plot(
    final_history["epoch"],
    final_history["val_original_iou"],
    marker="o",
    label="IoU",
)
axes[1].plot(
    final_history["epoch"],
    final_history["val_original_dice"],
    marker="o",
    label="Dice",
)
axes[1].set_title("Original-resolution validation")
axes[1].legend()

axes[2].plot(
    final_history["epoch"],
    final_history["learning_rate"],
    marker="o",
)
axes[2].set_title("Learning-rate schedule")

for axis in axes:
    axis.grid(alpha=0.25)
    axis.set_xlabel("Epoch")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "final_training_history.png", dpi=200)
plt.show()

## 17. Validation-only threshold calibration

In [ ]:
final_model, final_processor = load_segformer_checkpoint(
    str(FINAL_CHECKPOINT),
    device=DEVICE,
)

if should_run("calibrate"):
    STAGE_STATE.require("calibrate")
    STAGE_STATE.start("calibrate")
    try:
        validation_probability_rows = (
            collect_original_resolution_probabilities(
                final_model,
                final_processor,
                validation_df,
                image_size=CONFIG.data.image_size,
                resize_policy=CONFIG.data.resize_policy,
                device=DEVICE,
                batch_size=CONFIG.evaluation.batch_size,
                num_workers=CONFIG.data.num_workers,
            )
        )
        thresholds = np.round(
            np.arange(
                CONFIG.evaluation.threshold_minimum,
                CONFIG.evaluation.threshold_maximum + 1e-9,
                CONFIG.evaluation.threshold_step,
            ),
            4,
        ).tolist()
        SELECTED_THRESHOLD, threshold_frame = (
            calibrate_thresholds_from_probabilities(
                validation_probability_rows,
                thresholds=thresholds,
                empty_policy=CONFIG.evaluation.empty_mask_policy,
                output_csv=(
                    DIRECTORIES["calibration"]
                    / "validation_threshold_sweep.csv"
                ),
            )
        )
        threshold_metadata = {
            "validation_threshold": SELECTED_THRESHOLD,
            "selection_metric": (
                "mean per-image original-resolution validation IoU"
            ),
            "validation_images": len(validation_df),
            "test_split_used": False,
            "empty_mask_policy": CONFIG.evaluation.empty_mask_policy,
        }
        json_dump(
            threshold_metadata,
            DIRECTORIES["calibration"] / "selected_threshold.json",
        )
        json_dump(
            threshold_metadata,
            FINAL_CHECKPOINT / "selected_threshold.json",
        )
        STAGE_STATE.complete(
            "calibrate",
            evidence=[
                str(
                    DIRECTORIES["calibration"]
                    / "validation_threshold_sweep.csv"
                ),
                str(
                    DIRECTORIES["calibration"]
                    / "selected_threshold.json"
                ),
            ],
        )
    except Exception as exc:
        STAGE_STATE.fail("calibrate", str(exc))
        raise
else:
    threshold_frame = pd.read_csv(
        require_artifact(
            DIRECTORIES["calibration"]
            / "validation_threshold_sweep.csv",
            "threshold sweep",
        )
    )
    SELECTED_THRESHOLD = float(
        json_load(
            require_artifact(
                DIRECTORIES["calibration"]
                / "selected_threshold.json",
                "selected threshold",
            )
        )["validation_threshold"]
    )

display(
    threshold_frame.sort_values(
        ["iou", "dice"],
        ascending=False,
    ).head(10)
)
print("Selected validation threshold:", SELECTED_THRESHOLD)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    threshold_frame["threshold"],
    threshold_frame["iou"],
    marker="o",
    label="IoU",
)
plt.plot(
    threshold_frame["threshold"],
    threshold_frame["dice"],
    marker="o",
    label="Dice",
)
plt.axvline(
    SELECTED_THRESHOLD,
    linestyle="--",
    label=f"Selected={SELECTED_THRESHOLD:.2f}",
)
plt.title("Validation-only probability-threshold calibration")
plt.xlabel("Water probability threshold")
plt.ylabel("Mean per-image score")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "threshold_calibration.png", dpi=200)
plt.show()

## 18. Freeze selection and unlock the held-out test

In [ ]:
# Write an immutable model-selection lock before creating test_df.
weights_path = (
    FINAL_CHECKPOINT / "model.safetensors"
    if (FINAL_CHECKPOINT / "model.safetensors").exists()
    else FINAL_CHECKPOINT / "pytorch_model.bin"
)
selection_lock = {
    "timestamp_utc": utc_now_iso(),
    "git_commit": GIT_COMMIT,
    "split_registry_sha256": SPLIT_REGISTRY_SHA256,
    "selected_parameters": SELECTED_PARAMETERS,
    "final_seed": CONFIG.training.seed,
    "best_epoch": int(final_metadata["best_epoch"]),
    "checkpoint_sha256": sha256_file(weights_path),
    "validation_threshold": float(SELECTED_THRESHOLD),
    "selection_metric": (
        "mean per-image original-resolution validation IoU"
    ),
    "test_split_used_for_selection": False,
    "acceptance_criteria": ACCEPTANCE_CRITERIA,
}
SELECTION_LOCK_PATH = json_dump(
    selection_lock,
    OUTPUT_ROOT / "model_selection_lock.json",
)
print("Selection frozen:", SELECTION_LOCK_PATH)

# Only now is the held-out test dataframe materialized.
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)
assert len(test_df) == CONFIG.data.test_count
print("Held-out test unlocked:", len(test_df), "images")

## 19. Frozen final evaluation and full comparative inference

In [ ]:
if RUN_PROFILE == "full":
    evaluation_manifest = manifest.copy()
    evaluation_label = "final full 2,841-image inference"
else:
    # Smoke output is deliberately labelled and never presented as final.
    smoke_test = test_df.sample(
        n=min(PROFILE["evaluation_limit"], len(test_df)),
        random_state=CONFIG.training.seed,
    )
    smoke_validation = validation_df.sample(
        n=min(16, len(validation_df)),
        random_state=CONFIG.training.seed,
    )
    evaluation_manifest = pd.concat(
        [smoke_validation, smoke_test],
        ignore_index=True,
    )
    evaluation_label = "smoke subset inference — not a final result"

print(evaluation_label, len(evaluation_manifest))

In [ ]:
EVALUATION_CSV = (
    DIRECTORIES["evaluation"] / "segformer_v3_all_2841.csv"
    if RUN_PROFILE == "full"
    else DIRECTORIES["evaluation"] / "segformer_v3_smoke_subset.csv"
)

if should_run("evaluate"):
    STAGE_STATE.require("evaluate")
    STAGE_STATE.start("evaluate")
    try:
        v3_results, calibration_metrics = evaluate_manifest(
            final_model,
            final_processor,
            evaluation_manifest,
            image_size=CONFIG.data.image_size,
            resize_policy=CONFIG.data.resize_policy,
            threshold=SELECTED_THRESHOLD,
            device=DEVICE,
            batch_size=CONFIG.evaluation.batch_size,
            num_workers=CONFIG.data.num_workers,
            output_csv=EVALUATION_CSV,
            prediction_dir=(
                DIRECTORIES["evaluation"] / "predictions"
                if RUN_FULL_PREDICTION_EXPORT
                else None
            ),
            include_boundary_metrics=(
                CONFIG.evaluation.include_boundary_metrics
            ),
            boundary_tolerance=(
                CONFIG.evaluation.boundary_tolerance_pixels
            ),
            empty_policy=CONFIG.evaluation.empty_mask_policy,
            calibration_bins=CONFIG.evaluation.calibration_bins,
            calibration_output_dir=(
                DIRECTORIES["evaluation"] / "calibration"
            ),
        )
        json_dump(
            calibration_metrics,
            DIRECTORIES["evaluation"]
            / "probability_calibration_metrics.json",
        )
        STAGE_STATE.complete(
            "evaluate",
            evidence=[
                str(EVALUATION_CSV),
                str(
                    DIRECTORIES["evaluation"]
                    / "probability_calibration_metrics.json"
                ),
            ],
        )
    except Exception as exc:
        STAGE_STATE.fail("evaluate", str(exc))
        raise
else:
    v3_results = pd.read_csv(
        require_artifact(EVALUATION_CSV, "V3 evaluation table")
    )
    calibration_metrics = json_load(
        require_artifact(
            DIRECTORIES["evaluation"]
            / "probability_calibration_metrics.json",
            "probability calibration metrics",
        )
    )

display(v3_results.head())
display(pd.Series(calibration_metrics, name="value").to_frame())

In [ ]:
summary_by_split = summarize_results(v3_results)
summary_by_split.to_csv(
    DIRECTORIES["evaluation"] / "summary_by_split.csv",
    index=False,
)
display(summary_by_split)

if RUN_PROFILE == "full":
    test_v3 = v3_results[v3_results["split"] == "test"].copy()
    if len(test_v3) != 421:
        raise AssertionError(f"Expected 421 test rows, found {len(test_v3)}")
else:
    test_v3 = v3_results[v3_results["split"] == "test"].copy()

# Macro metrics treat each image equally. Pooled/global metrics aggregate pixels
# through the summed confusion matrix. Both are reported because water-body
# area varies substantially across images.
metric_columns = [
    "iou",
    "dice",
    "precision",
    "recall",
    "specificity",
    "pixel_accuracy",
    "balanced_accuracy",
    "mcc",
    "cohen_kappa",
    "boundary_f1",
    "boundary_iou",
    "hd95",
    "assd",
    "water_fraction_error",
    "model_forward_latency_ms",
]
test_metrics = {
    metric: float(test_v3[metric].mean())
    for metric in metric_columns
    if metric in test_v3
}

pooled = {
    key: int(test_v3[key].sum())
    for key in ("tp", "fp", "fn", "tn")
}
tp, fp, fn, tn = (
    pooled["tp"],
    pooled["fp"],
    pooled["fn"],
    pooled["tn"],
)
test_metrics.update(
    {
        "global_iou": float(tp / (tp + fp + fn))
        if (tp + fp + fn)
        else 1.0,
        "global_dice": float((2 * tp) / (2 * tp + fp + fn))
        if (2 * tp + fp + fn)
        else 1.0,
        "global_precision": float(tp / (tp + fp))
        if (tp + fp)
        else 1.0,
        "global_recall": float(tp / (tp + fn))
        if (tp + fn)
        else 1.0,
        "global_specificity": float(tn / (tn + fp))
        if (tn + fp)
        else 1.0,
    }
)

# Calibration is reported on the frozen test subset, not on pooled train,
# validation, and test pixels.
test_calibration_metrics = calibration_metrics.get(
    "test",
    calibration_metrics.get("overall", {}),
)
test_metrics.update(
    {
        "images": int(len(test_v3)),
        "water_present_iou": float(
            test_v3.loc[test_v3["target_has_water"], "iou"].mean()
        ),
        "empty_mask_images": int(
            (~test_v3["target_has_water"]).sum()
        ),
        "empty_mask_accuracy": float(
            test_v3.loc[
                ~test_v3["target_has_water"],
                "empty_mask_correct",
            ].mean()
        )
        if (~test_v3["target_has_water"]).any()
        else float("nan"),
        "empty_mask_false_positive_rate": float(
            test_v3.loc[
                ~test_v3["target_has_water"],
                "empty_mask_false_positive",
            ].mean()
        )
        if (~test_v3["target_has_water"]).any()
        else float("nan"),
        **test_calibration_metrics,
    }
)
json_dump(
    test_metrics,
    DIRECTORIES["evaluation"] / "segformer_v3_test_metrics.json",
)
display(pd.Series(test_metrics, name="value").to_frame())

test_reliability_path = (
    DIRECTORIES["evaluation"] / "calibration" / "test_reliability.csv"
)
if test_reliability_path.exists():
    test_reliability = pd.read_csv(test_reliability_path)
    nonempty_bins = test_reliability[
        test_reliability["pixel_count"] > 0
    ]
    plt.figure(figsize=(7, 7))
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    plt.plot(
        nonempty_bins["mean_confidence"],
        nonempty_bins["empirical_water_frequency"],
        marker="o",
        label="SegFormer V3",
    )
    plt.xlabel("Mean predicted water probability")
    plt.ylabel("Observed water frequency")
    plt.title("Frozen test reliability diagram")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "test_reliability_diagram.png",
        dpi=200,
    )
    plt.show()


## 20. Fair historical A–D comparison

In [ ]:
def locate_historical_csv(filename):
    candidates = [
        REPOSITORY_DIR / "results" / "full" / filename,
        REPOSITORY_DIR / "results" / filename,
    ]
    candidates.extend(INPUT_ROOT.rglob(filename))
    return next((path for path in candidates if path.exists()), None)

historical_files = {
    "A — Zero-shot SAM": locate_historical_csv(
        "experiment_A_zero_shot_sam_all_2841.csv"
    ),
    "B — Fine-tuned SAM": locate_historical_csv(
        "experiment_B_finetuned_sam_all_2841.csv"
    ),
    "C — Original SegFormer": locate_historical_csv(
        "experiment_C_segformer_all_2841.csv"
    ),
    "D — SegFormer–SAM hybrid": locate_historical_csv(
        "experiment_D_auto_sam_all_2841.csv"
    ),
}
historical_files

In [ ]:
historical_comparison_rows = []
historical_test_frames = {}

for label, path in historical_files.items():
    if path is None:
        print("Missing historical evidence:", label)
        continue
    frame = pd.read_csv(path)
    frame["split"] = (
        frame["split"].astype(str).str.lower().replace({"val": "validation"})
    )

    selected_prompt = "automatic"
    if (
        "prompt_mode" in frame.columns
        and frame["prompt_mode"].nunique() > 1
    ):
        validation_rows = frame[frame["split"] == "validation"]
        prompt_scores = validation_rows.groupby("prompt_mode")["iou"].mean()
        if prompt_scores.empty:
            raise RuntimeError(
                f"{label} has multiple prompt modes but no validation rows."
            )
        selected_prompt = prompt_scores.idxmax()
        frame = frame[frame["prompt_mode"] == selected_prompt]
    elif "prompt_mode" in frame.columns and len(frame):
        selected_prompt = str(frame["prompt_mode"].iloc[0])

    test_rows = frame[frame["split"] == "test"].copy()
    if len(test_rows) != CONFIG.data.test_count:
        raise RuntimeError(
            f"{label}: expected {CONFIG.data.test_count} test rows after "
            f"validation-selected filtering, found {len(test_rows)}."
        )
    historical_test_frames[label] = test_rows
    historical_comparison_rows.append(
        {
            "model": label,
            "configuration_selected_on_validation": selected_prompt,
            "test_images": len(test_rows),
            "iou": test_rows["iou"].mean(),
            "dice": test_rows["dice"].mean(),
            "precision": test_rows["precision"].mean(),
            "recall": test_rows["recall"].mean(),
            "specificity": (
                test_rows["specificity"].mean()
                if "specificity" in test_rows
                else float("nan")
            ),
            "pixel_accuracy": (
                test_rows["pixel_accuracy"].mean()
                if "pixel_accuracy" in test_rows
                else float("nan")
            ),
            "boundary_f1": test_rows["boundary_f1"].mean(),
            "recorded_model_stage_latency_ms": (
                test_rows["latency_ms"].mean()
            ),
            "latency_scope": (
                "SAM stage only; lower bound"
                if label == "D — SegFormer–SAM hybrid"
                else "historical recorded model stage"
            ),
        }
    )

historical_comparison_rows.append(
    {
        "model": "E — Tuned SegFormer V3",
        "configuration_selected_on_validation": "Optuna + confirmation",
        "test_images": len(test_v3),
        "iou": test_v3["iou"].mean(),
        "dice": test_v3["dice"].mean(),
        "precision": test_v3["precision"].mean(),
        "recall": test_v3["recall"].mean(),
        "specificity": test_v3["specificity"].mean(),
        "pixel_accuracy": test_v3["pixel_accuracy"].mean(),
        "boundary_f1": test_v3["boundary_f1"].mean(),
        "recorded_model_stage_latency_ms": (
            test_v3["model_forward_latency_ms"].mean()
        ),
        "latency_scope": "V3 measured model forward pass",
    }
)
historical_comparison = pd.DataFrame(historical_comparison_rows)
historical_comparison.to_csv(
    DIRECTORIES["evaluation"] / "historical_comparison.csv",
    index=False,
)
display(historical_comparison)

In [ ]:
historical_comparison.set_index("model")[
    ["iou", "dice", "boundary_f1"]
].plot(kind="bar", figsize=(13, 6))
plt.title("Held-out test comparison with validation-selected configurations")
plt.ylabel("Mean per-image score")
plt.ylim(0, 1)
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "historical_model_comparison.png", dpi=200)
plt.show()

## 21. Paired statistical comparison against historical SegFormer

In [ ]:
paired_statistics = {}
original_segformer = historical_test_frames.get(
    "C — Original SegFormer"
)

if RUN_PROFILE == "full" and original_segformer is not None:
    paired = test_v3[["image_id", "iou"]].merge(
        original_segformer[["image_id", "iou"]],
        on="image_id",
        suffixes=("_v3", "_historical"),
        validate="one_to_one",
    )
    if len(paired) != 421:
        raise AssertionError(
            f"Expected 421 paired test rows, found {len(paired)}."
        )

    paired_statistics.update(
        paired_bootstrap_difference(
            paired["iou_v3"].to_numpy(),
            paired["iou_historical"].to_numpy(),
            iterations=CONFIG.evaluation.bootstrap_iterations,
            seed=CONFIG.training.seed,
        )
    )
    paired_statistics.update(
        wilcoxon_paired(
            paired["iou_v3"].to_numpy(),
            paired["iou_historical"].to_numpy(),
        )
    )
    json_dump(
        paired_statistics,
        DIRECTORIES["statistics"] / "paired_comparison.json",
    )
    paired.to_csv(
        DIRECTORIES["statistics"] / "paired_iou_rows.csv",
        index=False,
    )
    display(pd.Series(paired_statistics, name="value").to_frame())

    plt.figure(figsize=(8, 6))
    plt.scatter(
        paired["iou_historical"],
        paired["iou_v3"],
        s=14,
        alpha=0.55,
    )
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("Historical SegFormer IoU")
    plt.ylabel("SegFormer V3 IoU")
    plt.title("Paired per-image test IoU")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "paired_iou_comparison.png", dpi=200)
    plt.show()
else:
    print("Paired final statistics require the full 421-image V3 test run.")

## 22. Performance slices, topology, and empty-mask behavior

In [ ]:
# Join dimensions and derive performance-relevant slices.
test_analysis = test_v3.merge(
    manifest[
        ["image_id", "width", "height", "water_fraction", "mask_path"]
    ],
    on="image_id",
    how="left",
    suffixes=("", "_registry"),
    validate="one_to_one",
)

# Derive topology and boundary complexity from ground-truth masks.
from scipy.ndimage import label as connected_components
from scipy.ndimage import binary_erosion

topology_rows = []
for row in test_analysis.itertuples(index=False):
    with Image.open(row.mask_path) as raw:
        target = np.asarray(raw.convert("L")) > 0
    _, components = connected_components(target)
    boundary = np.logical_xor(target, binary_erosion(target))
    area = int(target.sum())
    topology_rows.append(
        {
            "image_id": row.image_id,
            "connected_components": int(components),
            "topology_slice": (
                "no_water"
                if components == 0
                else "single_component"
                if components == 1
                else "multiple_components"
            ),
            "boundary_complexity": float(
                boundary.sum() / max(area, 1)
            ),
        }
    )

test_analysis = test_analysis.merge(
    pd.DataFrame(topology_rows),
    on="image_id",
    validate="one_to_one",
)
test_analysis = add_performance_slices(test_analysis)
test_analysis["boundary_complexity_slice"] = pd.qcut(
    test_analysis["boundary_complexity"],
    q=3,
    labels=["low", "medium", "high"],
    duplicates="drop",
)

slice_summary = summarize_slices(
    test_analysis,
    slice_columns=[
        "water_coverage_slice",
        "image_size_slice",
        "topology_slice",
        "boundary_complexity_slice",
    ],
    metric_columns=[
        "iou",
        "dice",
        "precision",
        "recall",
        "boundary_f1",
    ],
)
slice_summary.to_csv(
    DIRECTORIES["slices"] / "performance_slices.csv",
    index=False,
)
display(slice_summary)

In [ ]:
# Plot IoU by water coverage and topology.
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
slice_summary[
    slice_summary["slice_dimension"] == "water_coverage_slice"
].plot(
    x="slice_value",
    y="iou",
    kind="bar",
    legend=False,
    ax=axes[0],
)
axes[0].set_title("IoU by water coverage")
axes[0].set_ylabel("Mean IoU")
axes[0].tick_params(axis="x", rotation=30)

slice_summary[
    slice_summary["slice_dimension"] == "topology_slice"
].plot(
    x="slice_value",
    y="iou",
    kind="bar",
    legend=False,
    ax=axes[1],
)
axes[1].set_title("IoU by water-body topology")
axes[1].set_ylabel("Mean IoU")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "performance_slices.png", dpi=200)
plt.show()

## 23. Success, median, failure, false-positive, and false-negative examples

In [ ]:
def choose_qualitative_examples(frame):
    ordered = frame.sort_values("iou").reset_index(drop=True)
    selections = [
        ("worst_1", ordered.iloc[0]),
        ("worst_2", ordered.iloc[min(1, len(ordered) - 1)]),
        ("median", ordered.iloc[len(ordered) // 2]),
        ("best_2", ordered.iloc[max(0, len(ordered) - 2)]),
        ("best_1", ordered.iloc[-1]),
    ]
    empty_fp = frame[
        frame["empty_mask_false_positive"].astype(bool)
    ].sort_values("predicted_water_fraction", ascending=False)
    if len(empty_fp):
        selections.append(("empty_false_positive", empty_fp.iloc[0]))
    low_recall = frame.sort_values("recall").iloc[0]
    selections.append(("lowest_recall", low_recall))
    return selections

qualitative = choose_qualitative_examples(test_v3)
lookup = manifest.set_index("image_id")
fig, axes = plt.subplots(
    len(qualitative),
    4,
    figsize=(17, 4 * len(qualitative)),
)
if len(qualitative) == 1:
    axes = np.expand_dims(axes, 0)

for row_axes, (label, result) in zip(axes, qualitative):
    info = lookup.loc[str(result["image_id"])]
    with Image.open(info["image_path"]) as raw:
        image = np.asarray(raw.convert("RGB"))
    with Image.open(info["mask_path"]) as raw:
        target = (np.asarray(raw.convert("L")) > 0).astype(np.uint8)
    prediction_path = Path(result["prediction_path"])
    with Image.open(prediction_path) as raw:
        prediction = (
            np.asarray(raw.convert("L")) > 0
        ).astype(np.uint8)

    error = np.zeros((*target.shape, 3), dtype=np.uint8)
    error[(prediction == 1) & (target == 1)] = (255, 255, 255)
    error[(prediction == 1) & (target == 0)] = (255, 0, 0)
    error[(prediction == 0) & (target == 1)] = (0, 80, 255)

    row_axes[0].imshow(image)
    row_axes[0].set_title(f"{label}: {result['image_id']}")
    row_axes[1].imshow(target, cmap="gray")
    row_axes[1].set_title("Ground truth")
    row_axes[2].imshow(prediction, cmap="gray")
    row_axes[2].set_title(
        f"IoU={result['iou']:.3f} | "
        f"P={result['precision']:.3f} | R={result['recall']:.3f}"
    )
    row_axes[3].imshow(error)
    row_axes[3].set_title("White TP | red FP | blue FN")
    for axis in row_axes:
        axis.axis("off")

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "qualitative_success_failure_analysis.png",
    dpi=200,
)
plt.show()

## 24. Rigorous latency, throughput, and memory benchmark

In [ ]:
benchmark_example = test_df.sample(
    n=1,
    random_state=CONFIG.training.seed,
).iloc[0]

model_latency = benchmark_model_forward(
    final_model,
    final_processor,
    benchmark_example["image_path"],
    image_size=CONFIG.data.image_size,
    resize_policy=CONFIG.data.resize_policy,
    device=DEVICE,
    warmup_runs=5,
    timed_runs=50,
)
display(pd.Series(model_latency, name="value").to_frame())

## 25. Production inference, structured logs, and end-to-end latency

In [ ]:
# Create a minimal integrity manifest before the full registry record. The
# predictor validates the actual weights against this expected digest.
INTEGRITY_MANIFEST = DIRECTORIES["registry"] / "checkpoint_integrity.json"
json_dump(
    {
        "model_version": "segformer-v3.0.0",
        "checkpoint_sha256": selection_lock["checkpoint_sha256"],
        "validation_threshold": SELECTED_THRESHOLD,
        "image_size": CONFIG.data.image_size,
        "resize_policy": CONFIG.data.resize_policy,
    },
    INTEGRITY_MANIFEST,
)

INFERENCE_DIR = DIRECTORIES["production_inference"]
inference_sample = test_df.sample(
    n=1,
    random_state=CONFIG.training.seed,
).iloc[0]
input_copy = INFERENCE_DIR / Path(inference_sample["image_path"]).name
shutil.copy2(inference_sample["image_path"], input_copy)

if should_run("inference"):
    STAGE_STATE.require("inference")
    STAGE_STATE.start("inference")
    try:
        cold_started = time.perf_counter()
        predictor = SegFormerPredictor(
            FINAL_CHECKPOINT,
            selected_model_path=INTEGRITY_MANIFEST,
            image_size=CONFIG.data.image_size,
            resize_policy=CONFIG.data.resize_policy,
            device=DEVICE,
            log_path=INFERENCE_DIR / "inference.jsonl",
            model_version="segformer-v3.0.0",
            warmup_runs=1,
        )
        cold_start_ms = (time.perf_counter() - cold_started) * 1000.0

        mask, probability, inference_metadata = predictor.predict(
            input_copy
        )
        mask_path = predictor.save_mask(
            mask,
            INFERENCE_DIR / "predicted_water_mask.png",
        )
        overlay_path = predictor.save_overlay(
            input_copy,
            mask,
            INFERENCE_DIR / "predicted_water_overlay.png",
        )

        with Image.open(input_copy) as raw_input:
            input_width, input_height = raw_input.size
        with Image.open(mask_path) as raw_mask:
            saved_mask = np.asarray(raw_mask.convert("L"))

        assert saved_mask.shape == (input_height, input_width)
        assert set(np.unique(saved_mask).tolist()).issubset({0, 255})
        assert float(mask.mean()) == inference_metadata[
            "predicted_water_fraction"
        ]

        STAGE_STATE.complete(
            "inference",
            evidence=[
                str(mask_path),
                str(overlay_path),
                str(INFERENCE_DIR / "inference.jsonl"),
            ],
        )
    except Exception as exc:
        STAGE_STATE.fail("inference", str(exc))
        append_failure(
            FAILURE_LEDGER,
            stage="inference",
            error=exc,
            configuration={
                "checkpoint": str(FINAL_CHECKPOINT),
                "threshold": SELECTED_THRESHOLD,
            },
        )
        raise
else:
    mask_path = require_artifact(
        INFERENCE_DIR / "predicted_water_mask.png",
        "production mask",
    )
    overlay_path = require_artifact(
        INFERENCE_DIR / "predicted_water_overlay.png",
        "production overlay",
    )
    log_rows = [
        json.loads(line)
        for line in require_artifact(
            INFERENCE_DIR / "inference.jsonl",
            "inference log",
        ).read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    inference_metadata = log_rows[-1]
    cold_start_ms = float("nan")
    with Image.open(mask_path) as raw_mask:
        saved_mask = np.asarray(raw_mask.convert("L"))
    mask = (saved_mask > 0).astype(np.uint8)

print("Cold model-and-processor startup ms:", cold_start_ms)
display(pd.Series(inference_metadata, name="value").to_frame())

In [ ]:
# Warm end-to-end request latency includes preprocessing, model forward, and
# postprocessing. It is intentionally reported separately from model-only time.
if should_run("inference"):
    end_to_end_records = []
    for run_index in range(50):
        _, _, metadata = predictor.predict(
            input_copy,
            request_id=f"benchmark-{run_index:03d}",
        )
        end_to_end_records.append(metadata)

    end_to_end_frame = pd.DataFrame(end_to_end_records)
    end_to_end_frame.to_csv(
        INFERENCE_DIR / "end_to_end_latency.csv",
        index=False,
    )
else:
    end_to_end_frame = pd.read_csv(
        require_artifact(
            INFERENCE_DIR / "end_to_end_latency.csv",
            "end-to-end latency table",
        )
    )

end_to_end_latency = {
    "cold_start_ms": float(cold_start_ms),
    "p50_end_to_end_ms": float(
        end_to_end_frame["total_ms"].quantile(0.50)
    ),
    "p95_end_to_end_ms": float(
        end_to_end_frame["total_ms"].quantile(0.95)
    ),
    "mean_end_to_end_ms": float(
        end_to_end_frame["total_ms"].mean()
    ),
}
latency_metrics = {
    **model_latency,
    **end_to_end_latency,
}
json_dump(
    latency_metrics,
    INFERENCE_DIR / "latency_summary.json",
)
display(pd.Series(latency_metrics, name="value").to_frame())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
with Image.open(input_copy) as raw:
    axes[0].imshow(raw.convert("RGB"))
axes[0].set_title("Production input")
axes[1].imshow(saved_mask, cmap="gray")
axes[1].set_title(
    f"Binary mask | water={(saved_mask > 0).mean():.3f}"
)
with Image.open(overlay_path) as raw:
    axes[2].imshow(raw.convert("RGB"))
axes[2].set_title("Saved prediction overlay")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "production_inference_evidence.png",
    dpi=200,
)
plt.show()

inference_evidence = validate_inference_evidence(
    mask_path=mask_path,
    overlay_path=overlay_path,
    log_path=INFERENCE_DIR / "inference.jsonl",
)
if not inference_evidence["complete"]:
    raise RuntimeError(f"Inference evidence failed: {inference_evidence}")
display(pd.Series(inference_evidence, name="value").to_frame())

## 26. Portable model registry, dataset card, and model card

In [ ]:
# Build the production-candidate record from validation-selected and frozen-test
# evidence. Kaggle-local paths are kept separately from portable artifact paths.
hpo_best = json_load(
    DIRECTORIES["hpo"] / "best_trial.json"
)
validation_best = threshold_frame.sort_values(
    ["iou", "dice"],
    ascending=False,
).iloc[0]

deployment_status = (
    "production-candidate; Docker runtime validation pending"
    if RUN_PROFILE == "full"
    else "smoke-test artifact; not a production candidate"
)
model_record = build_model_record(
    model_version="segformer-v3.0.0",
    model_name="water-segformer-b0",
    base_model=MODEL_SPEC.model_id,
    checkpoint_dir=FINAL_CHECKPOINT,
    artifact_relative_path=(
        "artifacts/checkpoints/segformer_best"
    ),
    mlflow_artifact_uri=(
        final_metadata.get("mlflow_artifact_uri") or ""
    ),
    github_release_uri="",
    split_registry_sha256=SPLIT_REGISTRY_SHA256,
    git_commit=GIT_COMMIT,
    hpo_study=CONFIG.hpo.study_name,
    hpo_study_fingerprint=hpo_best["study_fingerprint"],
    final_parameters=SELECTED_PARAMETERS,
    best_epoch=int(final_metadata["best_epoch"]),
    validation_threshold=float(SELECTED_THRESHOLD),
    validation_metrics={
        "iou": float(validation_best["iou"]),
        "dice": float(validation_best["dice"]),
    },
    test_metrics=test_metrics,
    latency_metrics=latency_metrics,
    deployment_status=deployment_status,
    docker_validation_status="pending",
)

MODEL_REGISTRY_CSV, SELECTED_MODEL_JSON = write_model_registry(
    model_record,
    DIRECTORIES["registry"] / "model_registry.csv",
    DIRECTORIES["registry"] / "selected_model.json",
)
display(pd.Series(model_record, name="value").to_frame())

In [ ]:
# Human-readable dataset card generated from the audited registry.
registry_payload = json_load(DATA_REGISTRY_PATH)
dataset_card = f"""
# Dataset Card — Satellite Images of Water Bodies

## Source
- Dataset: {registry_payload['dataset_name']}
- Source: {registry_payload['dataset_source']}
- Valid pairs: {registry_payload['total_pairs']}

## Split
- Train: {registry_payload['train_pairs']}
- Validation: {registry_payload['validation_pairs']}
- Test: {registry_payload['test_pairs']}
- Split origin: {registry_payload.get('split_origin', 'not recorded')}
- Split registry SHA-256: `{SPLIT_REGISTRY_SHA256}`

## Validation
- Image-mask pairing by case-insensitive filename stem
- Every file decoded and dimension-checked
- Exact SHA-256 duplicate leakage rejected
- Perceptual-hash cross-split audit exported separately
- Split performed before any materialized tiling

## Data limitations
- The maintained model uses RGB only; no NIR or SWIR bands are available to the model.
- Geographic region, acquisition date, atmosphere, season, and sensor metadata are not
  consistently available in the registry.
- Geographic out-of-distribution generalization is therefore not established.
- GeoTIFF metadata is preserved where present, but many samples may be non-georeferenced PNGs
  or TIFFs.

## Intended use
Binary water/non-water segmentation research and controlled inference demonstrations.
"""
(DATASET_CARD_PATH := DIRECTORIES["docs"] / "dataset_card.md").write_text(
    dataset_card.strip() + "\n",
    encoding="utf-8",
)

model_card = f"""
# Model Card — Water SegFormer V3

## Model
- Version: `{model_record['model_version']}`
- Base model: `{model_record['base_model']}`
- Classes: non-water (0), water (1)
- Input policy: aspect-ratio-preserving letterbox to {CONFIG.data.image_size} × {CONFIG.data.image_size}
- Output: original-resolution binary mask
- Checkpoint SHA-256: `{model_record['checkpoint_sha256']}`
- Split registry SHA-256: `{SPLIT_REGISTRY_SHA256}`

## Selection
- HPO objective: original-resolution validation IoU at threshold 0.50
- Same-code historical configuration included during confirmation
- Final threshold selected on validation only: `{SELECTED_THRESHOLD:.4f}`
- Final seed fixed before test evaluation: `{CONFIG.training.seed}`

## Held-out test
- Images: `{test_metrics.get('images')}`
- IoU: `{test_metrics.get('iou', float('nan')):.6f}`
- Dice: `{test_metrics.get('dice', float('nan')):.6f}`
- Precision: `{test_metrics.get('precision', float('nan')):.6f}`
- Recall: `{test_metrics.get('recall', float('nan')):.6f}`
- Boundary F1: `{test_metrics.get('boundary_f1', float('nan')):.6f}`

## Operations
- Model-forward p50/p95: `{latency_metrics['p50_model_forward_ms']:.3f}` /
  `{latency_metrics['p95_model_forward_ms']:.3f}` ms
- End-to-end p50/p95: `{latency_metrics['p50_end_to_end_ms']:.3f}` /
  `{latency_metrics['p95_end_to_end_ms']:.3f}` ms
- Deployment status: {deployment_status}

## Limitations
- RGB-only inference can confuse shadows, dark soil, cloud shadows, and narrow water channels.
- No geographic OOD benchmark is available.
- Empty-mask conventions are disclosed separately.
- Docker runtime validation remains a separate local validation requirement.
"""
(MODEL_CARD_PATH := DIRECTORIES["docs"] / "model_card.md").write_text(
    model_card.strip() + "\n",
    encoding="utf-8",
)
print(DATASET_CARD_PATH)
print(MODEL_CARD_PATH)

## 27. Deployment package and export–reload parity

In [ ]:
DEPLOYMENT_ROOT = DIRECTORIES["exports"] / "segformer_v3_deployment"
if DEPLOYMENT_ROOT.exists():
    shutil.rmtree(DEPLOYMENT_ROOT)
DEPLOYMENT_ROOT.mkdir(parents=True)

deployment_checkpoint = DEPLOYMENT_ROOT / "segformer_best"
shutil.copytree(FINAL_CHECKPOINT, deployment_checkpoint)
shutil.copy2(
    SELECTED_MODEL_JSON,
    DEPLOYMENT_ROOT / "selected_model.json",
)
shutil.copy2(
    DIRECTORIES["calibration"] / "selected_threshold.json",
    DEPLOYMENT_ROOT / "selected_threshold.json",
)
shutil.copy2(
    MODEL_CARD_PATH,
    DEPLOYMENT_ROOT / "model_card.md",
)
shutil.copy2(
    REPOSITORY_DIR / "configs" / "inference_v3.yaml",
    DEPLOYMENT_ROOT / "inference_v3.yaml",
)

deployment_readme = """
# SegFormer V3 deployment artifact

Required files:
- `segformer_best/`
- `selected_model.json`
- `selected_threshold.json`
- `inference_v3.yaml`
- `model_card.md`

The predictor validates the checkpoint SHA-256 against `selected_model.json`
before serving inference.
"""
(DEPLOYMENT_ROOT / "README.md").write_text(
    deployment_readme.strip() + "\n",
    encoding="utf-8",
)

DEPLOYMENT_ZIP = Path(
    shutil.make_archive(
        (
            "/kaggle/working/"
            f"aereo-water-segformer-v3-deployment{ARTIFACT_SUFFIX}"
        ),
        "zip",
        root_dir=DEPLOYMENT_ROOT,
    )
)
print("Deployment ZIP:", DEPLOYMENT_ZIP)

In [ ]:
# Extract into a fresh directory and prove prediction parity.
PARITY_ROOT = DIRECTORIES["exports"] / "roundtrip_extract"
if PARITY_ROOT.exists():
    shutil.rmtree(PARITY_ROOT)
PARITY_ROOT.mkdir(parents=True)
shutil.unpack_archive(DEPLOYMENT_ZIP, PARITY_ROOT)

roundtrip_predictor = SegFormerPredictor(
    PARITY_ROOT / "segformer_best",
    selected_model_path=PARITY_ROOT / "selected_model.json",
    image_size=CONFIG.data.image_size,
    resize_policy=CONFIG.data.resize_policy,
    device=DEVICE,
    log_path=INFERENCE_DIR / "roundtrip_inference.jsonl",
    model_version="segformer-v3.0.0",
    warmup_runs=1,
)
roundtrip_mask, _, roundtrip_metadata = roundtrip_predictor.predict(
    input_copy
)
if not np.array_equal(mask, roundtrip_mask):
    raise AssertionError(
        "Deployment export–reload prediction parity failed."
    )

parity_evidence = {
    "prediction_masks_equal": True,
    "checkpoint_hash_before": model_record["checkpoint_sha256"],
    "checkpoint_hash_after": roundtrip_predictor.checkpoint_sha256,
    "hashes_equal": (
        model_record["checkpoint_sha256"]
        == roundtrip_predictor.checkpoint_sha256
    ),
}
json_dump(
    parity_evidence,
    DIRECTORIES["exports"] / "roundtrip_parity.json",
)
display(pd.Series(parity_evidence, name="value").to_frame())

## 28. Complete MLflow evaluation record and optional W&B mirror

In [ ]:
# Add calibration, frozen test results, figures, model card, and selected-model
# metadata to a linked evaluation run. Training and evaluation evidence are
# separated but connected by parent_run_id.
import mlflow
from mlflow.models import infer_signature

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
evaluation_experiment = mlflow.get_experiment_by_name(
    CONFIG.tracking.mlflow_experiment_evaluation
)
if evaluation_experiment is None:
    evaluation_experiment_id = mlflow.create_experiment(
        CONFIG.tracking.mlflow_experiment_evaluation,
        artifact_location=MLFLOW_ARTIFACT_ROOT.as_uri(),
    )
else:
    evaluation_experiment_id = evaluation_experiment.experiment_id

with mlflow.start_run(
    experiment_id=evaluation_experiment_id,
    run_name="segformer_v3_frozen_evaluation",
    tags={
        "parent_training_run_id": (
            final_metadata.get("mlflow_run_id") or ""
        ),
        "git_commit": GIT_COMMIT,
        "split_registry_sha256": SPLIT_REGISTRY_SHA256,
        "test_split_frozen": "true",
    },
) as evaluation_run:
    mlflow.log_params(
        {
            "model_version": model_record["model_version"],
            "validation_threshold": SELECTED_THRESHOLD,
            "test_images": len(test_v3),
            "empty_mask_policy": (
                CONFIG.evaluation.empty_mask_policy
            ),
            "resize_policy": CONFIG.data.resize_policy,
        }
    )
    mlflow.log_metrics(
        {
            key: float(value)
            for key, value in test_metrics.items()
            if isinstance(value, (int, float))
            and np.isfinite(value)
        }
    )
    mlflow.log_metrics(
        {
            key: float(value)
            for key, value in latency_metrics.items()
            if isinstance(value, (int, float))
            and np.isfinite(value)
        }
    )
    for artifact in [
        DIRECTORIES["calibration"] / "validation_threshold_sweep.csv",
        DIRECTORIES["calibration"] / "selected_threshold.json",
        DIRECTORIES["evaluation"] / "segformer_v3_test_metrics.json",
        DIRECTORIES["evaluation"] / "historical_comparison.csv",
        MODEL_REGISTRY_CSV,
        SELECTED_MODEL_JSON,
        MODEL_CARD_PATH,
        DATASET_CARD_PATH,
        OUTPUT_ROOT / "model_selection_lock.json",
    ]:
        if Path(artifact).exists():
            mlflow.log_artifact(str(artifact), artifact_path="evidence")
    mlflow.log_artifacts(str(FIGURE_DIR), artifact_path="figures")

    # Log the Hugging Face checkpoint, processor, threshold, and registry as
    # the deployment source of truth.
    mlflow.log_artifacts(
        str(FINAL_CHECKPOINT),
        artifact_path="huggingface_checkpoint",
    )

    # Register an inference-compatible logits wrapper when the installed
    # MLflow/PyTorch versions support it. Failure is recorded without losing
    # the authoritative Hugging Face artifact.
    PYTORCH_FLAVOR_LOGGED = False
    try:
        class MLflowSegFormerLogits(torch.nn.Module):
            def __init__(self, wrapped_model):
                super().__init__()
                self.wrapped_model = wrapped_model

            def forward(self, pixel_values):
                return self.wrapped_model(
                    pixel_values=pixel_values
                ).logits

        final_model.to("cpu").eval()
        mlflow_model = MLflowSegFormerLogits(final_model).eval()
        input_example = np.zeros(
            (1, 3, CONFIG.data.image_size, CONFIG.data.image_size),
            dtype=np.float32,
        )
        with torch.inference_mode():
            example_output = (
                mlflow_model(torch.from_numpy(input_example))
                .detach()
                .cpu()
                .numpy()
            )
        signature = infer_signature(input_example, example_output)
        mlflow.pytorch.log_model(
            mlflow_model,
            artifact_path="pytorch_model",
            signature=signature,
            input_example=input_example,
        )
        PYTORCH_FLAVOR_LOGGED = True
    except Exception as exc:
        append_failure(
            FAILURE_LEDGER,
            stage="mlflow_pytorch_flavor",
            error=exc,
            root_cause=(
                "Optional MLflow PyTorch flavor compatibility failure"
            ),
            resolution=(
                "Use the logged Hugging Face checkpoint and portable model "
                "registry as the deployment artifacts."
            ),
        )
        print(
            "MLflow PyTorch flavor warning:",
            type(exc).__name__,
            exc,
        )
    EVALUATION_RUN_ID = evaluation_run.info.run_id

try:
    if not PYTORCH_FLAVOR_LOGGED:
        raise RuntimeError(
            "PyTorch flavor was not logged; model registration skipped."
        )
    registration = mlflow.register_model(
        model_uri=f"runs:/{EVALUATION_RUN_ID}/pytorch_model",
        name="water-segformer-b0",
    )
    print("MLflow registered model version:", registration.version)
except Exception as exc:
    append_failure(
        FAILURE_LEDGER,
        stage="mlflow_model_registration",
        error=exc,
        root_cause="MLflow registry operation failed",
        resolution=(
            "Portable registry and Hugging Face deployment artifact remain "
            "authoritative; inspect MLflow backend compatibility."
        ),
    )
    print("MLflow model registration warning:", type(exc).__name__, exc)

In [ ]:
# Optional W&B evaluation mirror. Failure does not invalidate the authoritative
# MLflow record, but it is written to the failure ledger.
if WANDB_MODE != "disabled":
    try:
        import wandb

        evaluation_wandb = wandb.init(
            project=CONFIG.tracking.wandb_project,
            name="segformer_v3_frozen_evaluation",
            group="water_segformer_evaluation",
            mode=WANDB_MODE,
            dir=str(WANDB_ROOT),
            config={
                "model_version": model_record["model_version"],
                "validation_threshold": SELECTED_THRESHOLD,
                "git_commit": GIT_COMMIT,
                "split_registry_sha256": SPLIT_REGISTRY_SHA256,
            },
            reinit=True,
        )
        evaluation_wandb.log(
            {
                **{
                    key: value
                    for key, value in test_metrics.items()
                    if isinstance(value, (int, float))
                    and np.isfinite(value)
                },
                **latency_metrics,
            }
        )
        evidence_artifact = wandb.Artifact(
            "segformer-v3-evaluation-evidence",
            type="evaluation",
        )
        evidence_artifact.add_file(str(MODEL_CARD_PATH))
        evidence_artifact.add_file(str(SELECTED_MODEL_JSON))
        evidence_artifact.add_file(
            str(
                DIRECTORIES["evaluation"]
                / "segformer_v3_test_metrics.json"
            )
        )
        evaluation_wandb.log_artifact(evidence_artifact)
        evaluation_wandb.finish()
    except Exception as exc:
        append_failure(
            FAILURE_LEDGER,
            stage="wandb_evaluation_mirror",
            error=exc,
            root_cause="Optional W&B mirror failure",
            resolution="Retain MLflow as the system of record.",
        )
        print("W&B mirror warning:", type(exc).__name__, exc)

In [ ]:
# Export a human-readable MLflow run table.
experiments = mlflow.search_experiments()
all_runs = []
for experiment in experiments:
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
    )
    if not runs.empty:
        runs.insert(0, "experiment_name", experiment.name)
        all_runs.append(runs)
mlflow_runs = (
    pd.concat(all_runs, ignore_index=True)
    if all_runs
    else pd.DataFrame()
)
mlflow_runs.to_csv(
    DIRECTORIES["tracking"] / "mlflow_runs.csv",
    index=False,
)
display(
    mlflow_runs[
        [
            column
            for column in [
                "experiment_name",
                "run_id",
                "status",
                "tags.mlflow.runName",
                "metrics.best_validation_iou",
                "metrics.iou",
                "start_time",
                "end_time",
            ]
            if column in mlflow_runs
        ]
    ].head(30)
)

## 29. FastAPI smoke test with real checkpoint

In [ ]:
# Release notebook-held models before starting a second model instance inside
# the FastAPI lifespan. Stage-resume runs may not have instantiated every name.
for variable_name in (
    "roundtrip_predictor",
    "predictor",
    "final_model",
    "final_processor",
):
    globals().pop(variable_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
if RUN_API_SMOKE_TEST and should_run("api_test"):
    STAGE_STATE.require("api_test")
    STAGE_STATE.start("api_test")
    try:
        os.environ["AEREO_CHECKPOINT"] = str(FINAL_CHECKPOINT)
        os.environ["AEREO_SELECTED_MODEL"] = str(SELECTED_MODEL_JSON)
        os.environ["AEREO_DEVICE"] = str(DEVICE)
        os.environ["AEREO_IMAGE_SIZE"] = str(CONFIG.data.image_size)
        os.environ["AEREO_RESIZE_POLICY"] = CONFIG.data.resize_policy
        os.environ["AEREO_LOG_PATH"] = str(
            DIRECTORIES["api_validation"] / "api_inference.jsonl"
        )
        os.environ["AEREO_MAX_UPLOAD_BYTES"] = "1024"
        os.environ["AEREO_MAX_IMAGE_PIXELS"] = "1000000"
        os.environ["AEREO_MAX_CONCURRENCY"] = "1"

        from fastapi.testclient import TestClient
        from aereo_water.api.app import create_app

        api_app = create_app()
        with TestClient(api_app) as client:
            health_response = client.get("/health")
            ready_response = client.get("/ready")
            metadata_response = client.get("/metadata")

            tiny_image = Image.new(
                "RGB",
                (32, 24),
                color=(40, 80, 120),
            )
            tiny_buffer = io.BytesIO()
            tiny_image.save(tiny_buffer, format="PNG")
            tiny_bytes = tiny_buffer.getvalue()
            if len(tiny_bytes) >= 1024:
                raise AssertionError(
                    "Generated valid API image exceeds smoke-test limit."
                )

            valid_response = client.post(
                "/segment",
                files={
                    "image": (
                        "tiny.png",
                        tiny_bytes,
                        "image/png",
                    )
                },
            )
            invalid_response = client.post(
                "/segment",
                files={
                    "image": (
                        "invalid.txt",
                        b"not an image",
                        "text/plain",
                    )
                },
            )
            missing_response = client.post("/segment")
            oversized_response = client.post(
                "/segment",
                files={
                    "image": (
                        "large.png",
                        b"x" * 2048,
                        "image/png",
                    )
                },
            )

        api_evidence = {
            "health_status": health_response.status_code,
            "ready_status": ready_response.status_code,
            "metadata_status": metadata_response.status_code,
            "valid_upload_status": valid_response.status_code,
            "valid_content_type": valid_response.headers.get(
                "content-type"
            ),
            "invalid_content_type_status": invalid_response.status_code,
            "missing_file_status": missing_response.status_code,
            "oversized_upload_status": oversized_response.status_code,
        }
        expected_api = {
            "health_status": 200,
            "ready_status": 200,
            "metadata_status": 200,
            "valid_upload_status": 200,
            "invalid_content_type_status": 415,
            "missing_file_status": 422,
            "oversized_upload_status": 413,
        }
        for key, expected in expected_api.items():
            if api_evidence[key] != expected:
                raise AssertionError(
                    f"{key}: {api_evidence[key]} != {expected}"
                )
        if "image/png" not in api_evidence["valid_content_type"]:
            raise AssertionError(api_evidence["valid_content_type"])

        (DIRECTORIES["api_validation"] / "valid_response.png").write_bytes(
            valid_response.content
        )
        json_dump(
            api_evidence,
            DIRECTORIES["api_validation"] / "api_smoke_test.json",
        )
        STAGE_STATE.complete(
            "api_test",
            evidence=[
                str(
                    DIRECTORIES["api_validation"]
                    / "api_smoke_test.json"
                ),
                str(
                    DIRECTORIES["api_validation"]
                    / "valid_response.png"
                ),
            ],
        )
    except Exception as exc:
        STAGE_STATE.fail("api_test", str(exc))
        append_failure(
            FAILURE_LEDGER,
            stage="api_test",
            error=exc,
            root_cause="FastAPI smoke-test failure",
        )
        raise
elif RUN_API_SMOKE_TEST:
    api_evidence = json_load(
        require_artifact(
            DIRECTORIES["api_validation"] / "api_smoke_test.json",
            "API smoke-test evidence",
        )
    )
else:
    api_evidence = {}
    print("API smoke test disabled.")
display(pd.Series(api_evidence, name="value").to_frame())

## 30. Repository compilation, unit tests, and CI-ready evidence

In [ ]:
compile_result = run_command(
    [sys.executable, "-m", "compileall", "-q", "src", "scripts"],
    cwd=REPOSITORY_DIR,
    check=False,
)
pytest_result = run_command(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=REPOSITORY_DIR,
    check=False,
)

test_evidence = {
    "compile_exit_code": compile_result.returncode,
    "pytest_exit_code": pytest_result.returncode,
    "compile_stdout": compile_result.stdout,
    "compile_stderr": compile_result.stderr,
    "pytest_stdout": pytest_result.stdout,
    "pytest_stderr": pytest_result.stderr,
}
json_dump(
    test_evidence,
    DIRECTORIES["tests"] / "repository_test_results.json",
)
(DIRECTORIES["tests"] / "pytest_output.txt").write_text(
    pytest_result.stdout + "\n" + pytest_result.stderr,
    encoding="utf-8",
)

if compile_result.returncode != 0:
    raise RuntimeError("Repository compilation failed.")
if pytest_result.returncode != 0:
    raise RuntimeError("Repository tests failed.")

print(pytest_result.stdout)

## 31. Acceptance criteria evaluated without retrospective editing

In [ ]:
historical_iou = float(
    ACCEPTANCE_CRITERIA["quality"][
        "historical_segformer_test_iou"
    ]
)
maximum_regression = float(
    ACCEPTANCE_CRITERIA["quality"][
        "maximum_allowed_mean_iou_regression"
    ]
)
criteria_rows = [
    {
        "criterion": "Quality: no material IoU regression",
        "threshold": historical_iou - maximum_regression,
        "observed": test_metrics.get("iou"),
        "passed": (
            test_metrics.get("iou", -np.inf)
            >= historical_iou - maximum_regression
        )
        if RUN_PROFILE == "full"
        else False,
        "final_evidence_required": RUN_PROFILE != "full",
    },
    {
        "criterion": "Robustness: empty-mask false-positive rate",
        "threshold": ACCEPTANCE_CRITERIA["robustness"][
            "maximum_empty_mask_false_positive_rate"
        ],
        "observed": test_metrics.get(
            "empty_mask_false_positive_rate"
        ),
        "passed": (
            test_metrics.get(
                "empty_mask_false_positive_rate",
                np.inf,
            )
            <= ACCEPTANCE_CRITERIA["robustness"][
                "maximum_empty_mask_false_positive_rate"
            ]
        )
        if RUN_PROFILE == "full"
        else False,
        "final_evidence_required": RUN_PROFILE != "full",
    },
    {
        "criterion": "Operational: p95 end-to-end latency",
        "threshold": ACCEPTANCE_CRITERIA["operational"][
            "maximum_p95_end_to_end_latency_ms"
        ],
        "observed": latency_metrics["p95_end_to_end_ms"],
        "passed": (
            latency_metrics["p95_end_to_end_ms"]
            <= ACCEPTANCE_CRITERIA["operational"][
                "maximum_p95_end_to_end_latency_ms"
            ]
        ),
        "final_evidence_required": False,
    },
    {
        "criterion": "Integrity: checkpoint hash",
        "threshold": "exact match",
        "observed": parity_evidence["hashes_equal"],
        "passed": bool(parity_evidence["hashes_equal"]),
        "final_evidence_required": False,
    },
    {
        "criterion": "API smoke test",
        "threshold": "all expected statuses",
        "observed": api_evidence,
        "passed": bool(
            api_evidence
            and api_evidence.get("valid_upload_status") == 200
        ),
        "final_evidence_required": False,
    },
]
acceptance_frame = pd.DataFrame(criteria_rows)
acceptance_frame.to_csv(
    OUTPUT_ROOT / "acceptance_criteria_results.csv",
    index=False,
)
display(acceptance_frame)

## 32. Artifact export, resume bundle, and checksums

In [ ]:
if should_run("export"):
    STAGE_STATE.require("export")
    STAGE_STATE.start("export")

    # Small GitHub evidence excludes model weights and large tracking artifacts.
    github_evidence = DIRECTORIES["exports"] / "github_evidence"
    if github_evidence.exists():
        shutil.rmtree(github_evidence)
    github_evidence.mkdir(parents=True)

    evidence_directories = [
        "registry",
        "figures",
        "calibration",
        "evaluation",
        "statistics",
        "slices",
        "sample_predictions",
        "production_inference",
        "api_validation",
        "tests",
        "docs",
    ]
    for name in evidence_directories:
        source = DIRECTORIES[name]
        if source.exists():
            shutil.copytree(
                source,
                github_evidence / name,
                ignore=shutil.ignore_patterns(
                    "predictions",
                    "*.safetensors",
                    "pytorch_model.bin",
                ),
            )

    for source in [
        OUTPUT_ROOT / "run_environment.json",
        OUTPUT_ROOT / "pip_freeze.txt",
        OUTPUT_ROOT / "model_selection_lock.json",
        OUTPUT_ROOT / "selected_final_parameters.json",
        OUTPUT_ROOT / "assignment_compliance.csv",
        OUTPUT_ROOT / "acceptance_criteria_results.csv",
        OUTPUT_ROOT / "failure_ledger.csv",
        OUTPUT_ROOT / "stage_state.json",
    ]:
        if source.exists():
            shutil.copy2(source, github_evidence / source.name)

    GITHUB_EVIDENCE_ZIP = Path(
        shutil.make_archive(
            (
                "/kaggle/working/"
                f"aereo-water-v3-github-evidence{ARTIFACT_SUFFIX}"
            ),
            "zip",
            root_dir=github_evidence,
        )
    )

    # Resume bundle keeps Optuna, MLflow DB + artifacts, W&B offline runs,
    # confirmation, stability, and final resumable state.
    resume_root = DIRECTORIES["exports"] / "resume_bundle"
    if resume_root.exists():
        shutil.rmtree(resume_root)
    resume_root.mkdir(parents=True)
    for name in [
        "tracking",
        "hpo",
        "confirmation",
        "stability",
        "final_training",
        "calibration",
        "registry",
    ]:
        source = DIRECTORIES[name]
        if source.exists():
            shutil.copytree(source, resume_root / name)
    for source in [
        OUTPUT_ROOT / "stage_state.json",
        OUTPUT_ROOT / "model_selection_lock.json",
        OUTPUT_ROOT / "selected_final_parameters.json",
        OUTPUT_ROOT / "run_environment.json",
    ]:
        if source.exists():
            shutil.copy2(source, resume_root / source.name)

    RESUME_ZIP = Path(
        shutil.make_archive(
            (
                "/kaggle/working/"
                f"aereo-water-v3-resume{ARTIFACT_SUFFIX}"
            ),
            "zip",
            root_dir=resume_root,
        )
    )

    checksum_targets = [
        GITHUB_EVIDENCE_ZIP,
        RESUME_ZIP,
        DEPLOYMENT_ZIP,
        weights_path,
        SPLIT_REGISTRY_PATH,
        SELECTED_MODEL_JSON,
    ]
    checksum_lines = [
        f"{sha256_file(path)}  {Path(path).name}"
        for path in checksum_targets
        if Path(path).exists()
    ]
    CHECKSUM_PATH = Path(
        "/kaggle/working/"
        f"AEREO_V3{ARTIFACT_SUFFIX}_SHA256SUMS.txt"
    )
    CHECKSUM_PATH.write_text(
        "\n".join(checksum_lines) + "\n",
        encoding="utf-8",
    )

    STAGE_STATE.complete(
        "export",
        evidence=[
            str(GITHUB_EVIDENCE_ZIP),
            str(RESUME_ZIP),
            str(DEPLOYMENT_ZIP),
            str(CHECKSUM_PATH),
        ],
    )
else:
    GITHUB_EVIDENCE_ZIP = Path(
        (
            "/kaggle/working/"
            f"aereo-water-v3-github-evidence{ARTIFACT_SUFFIX}.zip"
        )
    )
    RESUME_ZIP = Path(
        (
            "/kaggle/working/"
            f"aereo-water-v3-resume{ARTIFACT_SUFFIX}.zip"
        )
    )
    CHECKSUM_PATH = Path(
        "/kaggle/working/AEREO_V3_SHA256SUMS.txt"
    )

print("GitHub evidence:", GITHUB_EVIDENCE_ZIP)
print("Resume bundle:", RESUME_ZIP)
print("Deployment bundle:", DEPLOYMENT_ZIP)
print("Checksums:", CHECKSUM_PATH)

## 33. Evidence-derived final assignment compliance

In [ ]:
compliance = build_compliance_table(
    output_root=OUTPUT_ROOT,
    expected_completed_hpo_trials=CONFIG.hpo.completed_trials,
    expected_total_inference_rows=(
        2841 if RUN_PROFILE == "full" else len(evaluation_manifest)
    ),
    pytest_exit_code=pytest_result.returncode,
    docker_validation_complete=False,
    presentation_complete=False,
)
compliance.to_csv(
    OUTPUT_ROOT / "assignment_compliance.csv",
    index=False,
)
display(compliance)

incomplete = compliance[~compliance["complete"]]
print("Incomplete requirements:", len(incomplete))
display(incomplete)

## 34. Executive results card

In [ ]:
same_code_baseline_iou = (
    float(baseline_confirmation.iloc[0]["best_validation_iou"])
    if len(baseline_confirmation)
    else float("nan")
)
historical_test_iou = (
    float(
        historical_comparison.loc[
            historical_comparison["model"]
            == "C — Original SegFormer",
            "iou",
        ].iloc[0]
    )
    if (
        historical_comparison["model"]
        == "C — Original SegFormer"
    ).any()
    else float("nan")
)

executive_summary = {
    "run_profile": RUN_PROFILE,
    "dataset_pairs": len(manifest),
    "train_validation_test": "1991 / 429 / 421",
    "selected_model": model_record["model_name"],
    "selected_hyperparameters": SELECTED_PARAMETERS,
    "same_code_baseline_validation_iou": same_code_baseline_iou,
    "selected_validation_iou": float(validation_best["iou"]),
    "selected_validation_threshold": SELECTED_THRESHOLD,
    "historical_segformer_test_iou": historical_test_iou,
    "segformer_v3_test_iou": test_metrics.get("iou"),
    "segformer_v3_test_dice": test_metrics.get("dice"),
    "segformer_v3_boundary_f1": test_metrics.get("boundary_f1"),
    "paired_mean_iou_change": paired_statistics.get(
        "mean_difference"
    ),
    "paired_95_ci": (
        [
            paired_statistics.get("ci_lower"),
            paired_statistics.get("ci_upper"),
        ]
        if paired_statistics
        else None
    ),
    "p50_model_forward_ms": latency_metrics[
        "p50_model_forward_ms"
    ],
    "p95_model_forward_ms": latency_metrics[
        "p95_model_forward_ms"
    ],
    "p50_end_to_end_ms": latency_metrics[
        "p50_end_to_end_ms"
    ],
    "p95_end_to_end_ms": latency_metrics[
        "p95_end_to_end_ms"
    ],
    "checkpoint_sha256": model_record["checkpoint_sha256"],
    "deployment_status": model_record["deployment_status"],
    "docker_validation_status": "pending local Docker validation",
}
json_dump(
    executive_summary,
    OUTPUT_ROOT / "executive_summary.json",
)
display(pd.Series(executive_summary, name="value").to_frame())

## 35. Limitations, failures, and next-step recommendations

### Evidence limitations

- The model is RGB-only. NIR and SWIR information that is highly informative for water detection
  is unavailable to the maintained model.
- Geographic region and acquisition-time metadata are not consistently available; geographic
  out-of-distribution generalization is not established.
- Exact duplicate leakage is rejected. Perceptual-hash similarity is an audit, not proof of
  spatial independence.
- Boundary distances are measured in pixels unless valid geospatial resolution metadata is
  available.
- Historical SAM experiments remain useful comparative evidence but depend on an older
  experimental implementation and oracle prompt protocols.
- The historical `8.3 ms` SegFormer latency and the V3 API latency are not directly equivalent:
  one is model-stage timing and the other includes preprocessing and postprocessing.

### Required work outside Kaggle

1. Copy the executed evidence into the cleaned GitHub repository.
2. Publish the deployment artifact and SHA-256 checksum through a GitHub Release.
3. Build `Dockerfile.v3` using `docker-compose.v3.yml`.
4. Validate `/health`, `/ready`, `/metadata`, real `/segment`, invalid upload, oversized upload,
   restart, and post-restart inference locally.
5. Update the report and presentation using only the measured V3 outputs.
6. Keep the old four-experiment notebook under the historical research section.

### Recommended technical extensions

- multispectral Sentinel-2 bands, especially NIR and SWIR;
- geographic or scene-grouped splitting when source metadata becomes available;
- cloud, haze, shadow, and seasonal robustness testing;
- uncertainty-driven manual review;
- larger SegFormer variants only after latency and memory budgets are defined;
- ONNX, quantization, or TensorRT only when implemented and measured rather than merely proposed.

## Reporting rules

Use the exported evidence exactly as measured.

Do not claim:

- a globally optimal hyperparameter configuration;
- a test improvement when the paired confidence interval includes zero without explaining it;
- geographic generalization;
- end-to-end Docker validation until the external Docker tests pass;
- W&B completion when no offline or online run artifact exists;
- production readiness based solely on notebook inference.

A negative or statistically uncertain result is still valuable when the method, split firewall,
tracking, and limitations are transparent.